# SAM-CapsNet — Reproducibility Notebook
### Attention-Conditioned Capsule Networks for Video-Based Human Activity Recognition

Archived at https://doi.org/10.5281/zenodo.23254308

**How to run**
1. Set the two dataset paths in **Section 7**.
2. For a fast end-to-end check, set `QUICK_TEST = True` in Section 7 and choose *Restart & Run All*. Every section then uses one seed and two epochs; the numbers are not meaningful, but every cell is exercised.
3. Set `QUICK_TEST = False` and choose *Restart & Run All* for the full run. Results, figures and the split file lists are written to `results_final/`.

The UCF-101 sections (24–27) are the longest and come last. Set `RUN_UCF101 = False` in Section 7 to stop before them.

**Before running**
- Run the notebook from the same folder as earlier runs. The frame caches (`.cache_ucf11`, `.cache_ucf101_r1`) are keyed by each clip's path, so they are reused and the UCF-101 cache does not have to be rebuilt.
- Every cell's run time is appended to `results_final/cell_times.csv`. After the `QUICK_TEST` run (1 seed, 2 epochs), multiply the time of each training section by about 37 (15/2 epochs × 5 seeds) to estimate the full run.

**Two training routines are used, as stated in Section II-F of the paper.**
- *Margin-loss routine* (Sections 19–23 and 27): Tables 1, 2 and 4 and the capsule rows of Table 3.
- *Cross-entropy routine* (Sections 9, 11, 16 and 18): cross-entropy on capsule lengths with mixed precision. The paper uses it for the attention-placement study, the SAM-CapsNet confusion matrix, the orientation test and the single-clip figures.

| Paper item | Section |
|---|---|
| Supplementary Table S3 (class counts); split file lists | 8 |
| Table 1, CNN rows | 10 |
| Sabour-style baseline under cross-entropy (Section II-E) | 11 |
| Figure 5 | 12 |
| Figure 2 | 13 |
| Figure 3 and Supplementary Figure S1 | 14 |
| Figure 4 | 15 |
| Table 5 (orientation test) | 16 |
| Inference latency (Section III-F) | 17 |
| Attention-placement study (Section III-A) | 18 |
| Table 1, capsule rows; Table 4 (routing concentration) | 19 |
| Table 1 and Table 4, network with attention removed | 20 |
| Table 3, pre-trained ResNet-18 stem on UCF-11 | 21 |
| Supplementary Table S2 | 22 |
| Significance tests (Section III-A); Table 2 (per-class accuracy) | 23 |
| Table 3, ResNet-18 and MobileNetV2 on UCF-101 | 25 (learning rate 1e-3), 26 (learning rate 1e-4) |
| Table 3 and Section III-F, capsule models on UCF-101 | 27 and 27B |

## 1. Install Dependencies

In [ ]:
%pip install torch torchvision opencv-python matplotlib seaborn scikit-learn tqdm --quiet

## 2. Imports

In [ ]:
import os, cv2, json, math, random, warnings
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms

%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import StratifiedShuffleSplit

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ── Per-cell wall-clock log: <SAVE_DIR>/cell_times.csv ───────────────────────
# After the QUICK_TEST run, this file shows how long each section takes, which is
# used to estimate the full run (about 37x the quick time for seed-looped sections).
import time as _time
from IPython import get_ipython
_ip = get_ipython()
if _ip is not None and not getattr(_ip, '_sam_timer', False):
    _t0 = {}
    def _pre_cell(*args):
        _t0['t'] = _time.time()
    def _post_cell(*args):
        dt = _time.time() - _t0.get('t', _time.time())
        raw = ''
        if args and hasattr(args[0], 'info'):
            raw = getattr(args[0].info, 'raw_cell', '') or ''
        first = raw.strip().split('\n')[0][:70].replace('"', "'")
        out_dir = globals().get('SAVE_DIR', '.')
        try:
            os.makedirs(out_dir, exist_ok=True)
            with open(os.path.join(out_dir, 'cell_times.csv'), 'a') as fh:
                fh.write('%s,%.1f,"%s"\n' % (_time.strftime('%Y-%m-%d %H:%M:%S'), dt, first))
        except Exception:
            pass
        if dt > 60:
            print('[cell time] %.1f min' % (dt / 60))
    _ip.events.register('pre_run_cell', _pre_cell)
    _ip.events.register('post_run_cell', _post_cell)
    _ip._sam_timer = True


## 3. Dataset
`UCF11VideoDataset` — 16 frames sampled at a fixed 5 fps from the start of each video, resized to 112×112.
The first run builds a disk cache; later runs load from it. Videos are listed in sorted order, so the splits depend only on the seed and not on the file system.

In [ ]:
import hashlib, pickle

class UCF11VideoDataset(Dataset):

    VIDEO_EXTS = ('*.avi', '*.mp4', '*.mov', '*.mpg', '*.mpeg')

    def __init__(self, root_dir, sequence_length=16,
                 target_size=(224, 224), sample_fps=5.0,
                 augment=False, cache_dir='.cache',
                 class_subset=None):
        self.root_dir        = Path(root_dir)
        self.sequence_length = sequence_length
        self.target_size     = target_size
        self.sample_fps      = sample_fps
        self.augment         = augment
        self.cache_dir       = Path(cache_dir) if cache_dir else None
        self.class_subset    = [c.lower() for c in class_subset] if class_subset else None

        aug_tf = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(target_size),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
            transforms.RandomRotation(degrees=8),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])
        clean_tf = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(target_size),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])
        self.transform = aug_tf if augment else clean_tf

        self.samples      = []
        self.class_to_idx = {}
        self.idx_to_class = {}
        self._setup()

        if self.cache_dir:
            self.cache_dir.mkdir(parents=True, exist_ok=True)

    def _setup(self):
        if not self.root_dir.exists():
            raise FileNotFoundError("Dataset not found: {}".format(self.root_dir))
        all_dirs = sorted(d.name for d in self.root_dir.iterdir()
                          if d.is_dir() and not d.name.startswith(('.','__')))
        if self.class_subset is not None:
            classes = [d for d in all_dirs
                       if d.lower() in self.class_subset]
            missing = [c for c in self.class_subset
                       if c not in [x.lower() for x in classes]]
            if missing:
                print('WARNING: these classes not found in dataset: {}'.format(missing))
        else:
            classes = all_dirs
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        self.idx_to_class = {i: c for i, c in enumerate(classes)}
        print("Found {} classes: {}".format(len(classes), classes))
        for cls in classes:
            p = self.root_dir / cls
            vids = []
            for ext in self.VIDEO_EXTS:
                vids.extend(p.glob(ext))
            for sub in p.iterdir():
                if sub.is_dir() and sub.name != 'annotation':
                    for ext in self.VIDEO_EXTS:
                        vids.extend(sub.glob(ext))
            for vp in sorted(vids):   # fixed order: splits depend only on the seed
                self.samples.append((str(vp), self.class_to_idx[cls]))
            print("  {}: {} videos".format(cls, len(vids)))
        print("Total: {} videos".format(len(self.samples)))

    def _cache_key(self, path):
        subset_key = ",".join(sorted(self.class_subset)) if self.class_subset else "all"
        sig = "{}|{}|{}|{}|{}".format(
            path, self.sequence_length, self.sample_fps, str(self.target_size), subset_key)
        return hashlib.md5(sig.encode()).hexdigest()

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        if self.cache_dir and not self.augment:
            cache_file = self.cache_dir / (self._cache_key(path) + '.pkl')
            if cache_file.exists():
                try:
                    return pickle.loads(cache_file.read_bytes()), label
                except Exception:
                    pass

        frames = self._load_video(path)
        if frames is None:
            frames = torch.zeros(self.sequence_length, 3, *self.target_size)

        if self.cache_dir and not self.augment:
            try:
                cache_file = self.cache_dir / (self._cache_key(path) + '.pkl')
                cache_file.write_bytes(pickle.dumps(frames))
            except Exception:
                pass

        return frames, label

    def _load_video(self, path):
        try:
            cap = cv2.VideoCapture(path)
            if not cap.isOpened():
                return None
            native_fps = cap.get(cv2.CAP_PROP_FPS)
            if native_fps <= 0 or native_fps > 240:
                native_fps = 25.0
            total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            if total <= 0:
                cap.release()
                return None

            # Fixed-rate anchors at sample_fps
            step    = max(1, int(round(native_fps / self.sample_fps)))
            anchors = [min(i * step, total - 1) for i in range(self.sequence_length)]

            # Sequential read: one forward pass, no random seeking
            cap.set(cv2.CAP_PROP_POS_FRAMES, anchors[0])
            raw   = {}
            cur   = anchors[0]
            last  = anchors[-1]
            while cur <= last:
                ret, frame = cap.read()
                if ret:
                    raw[cur] = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                cur += 1
            cap.release()

            if not raw:
                return None

            frames = []
            for anchor in anchors[:self.sequence_length]:
                # Single clean frame — no averaging
                fi = min(raw.keys(), key=lambda k: abs(k - anchor))
                frames.append(self.transform(raw[fi]))

            while len(frames) < self.sequence_length:
                frames.append(frames[-1])
            return torch.stack(frames[:self.sequence_length])

        except Exception as e:
            print("Error loading {}: {}".format(path, e))
            return None

    def get_labels(self):
        return [s[1] for s in self.samples]

    def clear_cache(self):
        if self.cache_dir and self.cache_dir.exists():
            import shutil
            shutil.rmtree(self.cache_dir)
            self.cache_dir.mkdir()
            print("Cache cleared.")


def make_stratified_splits(dataset, train_r=0.70, val_r=0.15, seed=42):
    labels = np.array(dataset.get_labels())
    n = len(labels)
    sss1 = StratifiedShuffleSplit(1, test_size=1-train_r-val_r, random_state=seed)
    tv_idx, test_idx = next(sss1.split(np.zeros(n), labels))
    val_frac = val_r / (train_r + val_r)
    sss2 = StratifiedShuffleSplit(1, test_size=val_frac, random_state=seed)
    rel_tr, rel_val = next(sss2.split(np.zeros(len(tv_idx)), labels[tv_idx]))
    return tv_idx[rel_tr].tolist(), tv_idx[rel_val].tolist(), test_idx.tolist()


print("Dataset class defined (5fps fixed-rate, clean frames, disk cache).")

## 4. Model Architecture
`SAMCapsNet` — the proposed model. `sam_config` selects the attention branches.

In [ ]:
# ── SAM-CapsNet: Correct Sequential Multi-Scale Attention ────────────────────
#
# DESIGN: sequential CBAM-style multiplicative attention:
#   Step 1: Channel attention rescales each channel independently.
#           Important channels → weight near 1.0 (preserved)
#           Background channels → weight near 0.0 (suppressed 10-20x)
#           Net effect: CONTRAST ENHANCEMENT between discriminative
#                       and non-discriminative channels.
#
#   Step 2: Spatial attention applied to channel-refined features.
#           Action regions → weight near 1.0 (preserved)
#           Background regions → weight near 0.0 (suppressed)
#           Guided by temporal motion map to resolve the noisy-map
#           problem on small datasets.
#
#   Step 3: Temporal attention applied as an additive residual.
#           Motion is added as a residual (it contributes where motion is
#           present) instead of multiplicatively gating existing features.
#           F_tm = F_sp + alpha * (F_sp * Am)
#           where alpha is a learned scalar in (-1, 1) via tanh.
#           This allows AMPLIFICATION of motion-active features.
#
# Point Prompt: learnable spatial prior P added before spatial attention,
#   anchoring saliency to action-relevant regions.
#
# Learnable fusion weight lam: blends spatial and temporal maps.

class MultiScaleSAMAttention(nn.Module):
    """
    Sequential multiplicative attention: Channel -> Spatial (temporal-guided)
    -> Temporal residual.

    Channel and spatial use standard multiplicative CBAM-style attention
    (contrast enhancement, not uniform suppression).
    Temporal uses signed residual (tanh gate) to allow feature amplification.
    """
    def __init__(self, in_channels=256, reduction_ratio=8,
                 use_channel=True, use_spatial=True, use_point_prompt=True,
                 use_temporal=True, use_learnable_fusion=True):
        super().__init__()
        self.use_channel          = use_channel
        self.use_spatial          = use_spatial
        self.use_point_prompt     = use_point_prompt
        self.use_temporal         = use_temporal
        self.use_learnable_fusion = use_learnable_fusion
        r = max(1, in_channels // reduction_ratio)

        # ── Channel: CBAM-style squeeze-excitation ────────────────────────
        if use_channel:
            self.ch_gap  = nn.AdaptiveAvgPool2d(1)
            self.ch_gmp  = nn.AdaptiveMaxPool2d(1)   # max pool adds selectivity
            self.ch_fc1  = nn.Linear(in_channels, r, bias=False)
            self.ch_fc2  = nn.Linear(r, in_channels, bias=False)

        # ── Point prompt ──────────────────────────────────────────────────
        if use_point_prompt:
            self.point_prompt = nn.Parameter(torch.zeros(1, in_channels, 1, 1))
            nn.init.normal_(self.point_prompt, std=0.01)

        # ── Temporal: dual-scale motion map ───────────────────────────────
        if use_temporal:
            self.tm_fine   = nn.Sequential(
                nn.Conv2d(in_channels*2, r//2, 3, padding=1, bias=False),
                nn.BatchNorm2d(r//2), nn.ReLU(True))
            self.tm_coarse = nn.Sequential(
                nn.Conv2d(in_channels*2, r//2, 5, padding=2, bias=False),
                nn.BatchNorm2d(r//2), nn.ReLU(True))
            self.tm_fuse   = nn.Sequential(
                nn.Conv2d(r, 1, 1, bias=False), nn.Sigmoid())
            self.tm_drop   = nn.Dropout2d(p=0.1)
            # Signed gate: tanh allows amplification of motion features
            self.alpha_tm  = nn.Parameter(torch.tensor(0.1))

        # ── Spatial: guided by temporal map ───────────────────────────────
        if use_spatial:
            extra = 1 if use_temporal else 0
            self.sp_guide = nn.Conv2d(in_channels + extra, in_channels,
                                       1, bias=False)
            self.sp_conv  = nn.Sequential(
                nn.Conv2d(in_channels, r, 1, bias=False),
                nn.BatchNorm2d(r), nn.ReLU(True),
                nn.Conv2d(r, r, 3, padding=1, bias=False),
                nn.BatchNorm2d(r), nn.ReLU(True),
                nn.Conv2d(r, 1, 1, bias=False), nn.Sigmoid())
            self.sp_drop  = nn.Dropout2d(p=0.1)

        if use_learnable_fusion:
            self.lam = nn.Parameter(torch.tensor(0.5))
        else:
            self.register_buffer('lam', torch.tensor(0.5))

    def forward(self, x, prev_x=None, return_maps=False):
        B, C, H, W = x.shape
        out = x
        a_m, a_s, ac = None, None, None

        # Step 1: Channel attention (multiplicative, CBAM-style)
        if self.use_channel:
            avg = self.ch_gap(x).view(B, C)
            mx  = self.ch_gmp(x).view(B, C)
            # Shared FC on both poolings, summed
            ac  = torch.sigmoid(
                self.ch_fc2(F.relu(self.ch_fc1(avg))) +
                self.ch_fc2(F.relu(self.ch_fc1(mx)))
            ).view(B, C, 1, 1)
            out = out * ac   # multiplicative: contrast enhancement

        # Step 2: Temporal motion map
        if self.use_temporal and prev_x is not None:
            cat = torch.cat([x, prev_x], dim=1)
            a_m = self.tm_fuse(
                torch.cat([self.tm_fine(cat), self.tm_coarse(cat)], dim=1))
            if self.training:
                a_m = self.tm_drop(a_m)

        # Step 3: Spatial attention (applied to channel-refined features)
        if self.use_spatial:
            inp = out + self.point_prompt if self.use_point_prompt else out
            if a_m is not None:
                inp = self.sp_guide(torch.cat([inp, a_m], dim=1))
            a_s = self.sp_conv(inp)
            if self.training:
                a_s = self.sp_drop(a_s)
            out = out * a_s   # multiplicative: spatial focus

        # Step 4: Temporal residual (signed gate allows amplification)
        if a_m is not None:
            alpha = torch.tanh(self.alpha_tm)   # in (-1, 1)
            out   = out + alpha * (out * a_m)   # amplify motion regions

        # Build visualisation maps (reuse cached tensors)
        if return_maps:
            lam = self.lam.clamp(0.0, 1.0) if hasattr(self, 'lam') else torch.tensor(0.5)
            if a_m is not None and a_s is not None:
                a_f = lam * a_s + (1.0 - lam) * a_m
            elif a_s is not None:
                a_f = a_s
            elif a_m is not None:
                a_f = a_m
            else:
                a_f = torch.ones(B, 1, H, W, device=x.device)
            if ac is None:
                ac = torch.ones(B, C, 1, 1, device=x.device)
            return out, a_f, ac

        return out, None, None

    def gate_values(self):
        """For logging/monitoring during training."""
        gv = {}
        if self.use_temporal and hasattr(self, 'alpha_tm'):
            gv['alpha_tm'] = torch.tanh(self.alpha_tm)
        return gv


class PrimaryCaps(nn.Module):
    def __init__(self, in_channels=256, out_capsules=32, out_dim=8):
        super().__init__()
        self.out_capsules, self.out_dim = out_capsules, out_dim
        self.caps_conv = nn.Conv2d(in_channels, out_capsules * out_dim,
                                   kernel_size=3, padding=1)

    def squash(self, v, eps=1e-8):
        n = v.norm(dim=-1, keepdim=True)
        return (n**2 / (1 + n**2)) * v / (n + eps)

    def forward(self, x):
        B = x.size(0)
        x = self.caps_conv(x)
        H, W = x.shape[2], x.shape[3]
        x = x.view(B, self.out_capsules, self.out_dim, H, W)
        x = x.permute(0, 3, 4, 1, 2).contiguous().view(B, -1, self.out_dim)
        return self.squash(x)


class ActivityCaps(nn.Module):
    def __init__(self, in_capsules, in_dim=8, out_capsules=11, out_dim=16,
                 routing_iters=3):
        super().__init__()
        self.out_capsules  = out_capsules
        self.out_dim       = out_dim
        self.routing_iters = routing_iters
        self.W = nn.Parameter(
            0.01 * torch.randn(1, in_capsules, out_capsules, out_dim, in_dim))

    def squash(self, v, eps=1e-8):
        n = v.norm(dim=-1, keepdim=True)
        return (n**2 / (1 + n**2)) * v / (n + eps)

    def forward(self, u):
        B = u.shape[0]
        W = self.W.expand(B, -1, -1, -1, -1)
        u_hat = torch.matmul(W, u.unsqueeze(2).unsqueeze(-1)).squeeze(-1)
        b = torch.zeros(B, u.shape[1], self.out_capsules, 1, device=u.device)
        for r in range(self.routing_iters):
            c = F.softmax(b, dim=2)
            s = (c * u_hat).sum(dim=1)
            v = self.squash(s)
            if r < self.routing_iters - 1:
                b = b + (u_hat * v.unsqueeze(1)).sum(dim=-1, keepdim=True)
        return v


class SAMCapsNet(nn.Module):
    FEAT_H, FEAT_W = 7, 7   # 112x112 input -> 7x7 after 4 downsampling blocks
    PRIM_CAPS, PRIM_DIM = 16, 8  # 16 caps at 7x7 -> 784 in_caps (vs 6272)

    def __init__(self, num_classes=11, sam_config=None):
        super().__init__()
        self.num_classes = num_classes
        self.use_sam     = (sam_config is not None)

        self.conv = nn.Sequential(
            nn.Conv2d(3,   64,  7, stride=2, padding=3),
            nn.BatchNorm2d(64),  nn.ReLU(True),
            nn.MaxPool2d(3, stride=2, padding=1),
            nn.Conv2d(64,  128, 3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(True),
            nn.MaxPool2d(2, stride=2),
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(True),
            nn.MaxPool2d(2, stride=2),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(True),
        )

        if self.use_sam:
            cfg = dict(use_channel=True, use_spatial=True,
                       use_point_prompt=True, use_temporal=True,
                       use_learnable_fusion=True)
            cfg.update(sam_config)
            self.attention = MultiScaleSAMAttention(in_channels=256, **cfg)

        in_caps = self.PRIM_CAPS * self.FEAT_H * self.FEAT_W  # 6272
        self.pre_caps_norm = nn.LayerNorm([256, self.FEAT_H, self.FEAT_W])  # dynamic
        self.primary  = PrimaryCaps(256, self.PRIM_CAPS, self.PRIM_DIM)
        self.activity = ActivityCaps(in_caps, self.PRIM_DIM, num_classes, 16)

    def forward(self, x, return_attention=False):
        B, T, C, H, W = x.shape

        # Batch all frames through conv in one call
        x_flat     = x.view(B * T, C, H, W)
        feats_flat = self.conv(x_flat)
        feats      = feats_flat.view(B, T, 256, self.FEAT_H, self.FEAT_W)

        attended, sp_maps, ch_maps = [], [], []
        for t in range(T):
            if self.use_sam:
                prev = feats[:, t-1] if t > 0 else None
                f, sa, ca = self.attention(
                    feats[:, t], prev, return_maps=return_attention)
                attended.append(f)
                if return_attention and sa is not None:
                    sp_maps.append(sa)
                    ch_maps.append(ca)
            else:
                attended.append(feats[:, t])

        agg  = torch.stack(attended, dim=1).mean(dim=1)
        agg  = self.pre_caps_norm(agg)
        caps = self.primary(agg)
        out  = self.activity(caps)

        if return_attention and self.use_sam and sp_maps:
            return out, {
                'spatial':      torch.stack(sp_maps, dim=1),
                'channel':      torch.stack(ch_maps, dim=1),
                'feature_size': (self.FEAT_H, self.FEAT_W),
                'input_size':   (H, W)
            }
        return out

    def gate_sparsity_loss(self, lam=0.0):
        # No sparsity loss needed with multiplicative attention
        # (branches can be zeroed by the attention maps themselves)
        return torch.tensor(0.0, requires_grad=True, device=
                            next(self.parameters()).device)


print("SAMCapsNet with CBAM-style sequential multiplicative attention defined.")

_d = torch.zeros(2, 16, 3, 112, 112)
_m = SAMCapsNet(num_classes=11, sam_config={})
with torch.no_grad():
    _o = _m(_d)
assert tuple(_o.shape) == (2, 11, 16)
print("Forward pass OK:", tuple(_o.shape))

# Verify no suppression-only behaviour
print()
print("Channel attention sample output (should have values near 1.0 for")
print("important channels, not uniformly small):")
_f = torch.randn(2, 256, 14, 14)
_att = _m.attention
with torch.no_grad():
    avg = _att.ch_gap(_f).view(2, 256)
    mx  = _att.ch_gmp(_f).view(2, 256)
    ac  = torch.sigmoid(_att.ch_fc2(F.relu(_att.ch_fc1(avg))) +
                        _att.ch_fc2(F.relu(_att.ch_fc1(mx))))
print("  ac min={:.3f}  max={:.3f}  mean={:.3f}".format(
      ac.min().item(), ac.max().item(), ac.mean().item()))
print("  (min near 0, max near 1 = contrast enhancement working)")

del _d, _m, _o, _f, _att, ac

## 4B. CNN baseline — Table 1 CNN rows, Figure 5(a)
`BaselineCNN` uses a comparable convolutional encoder with a pooled softmax classifier, with or without channel and spatial attention.

In [ ]:
class BaselineCNN(nn.Module):
    """
    Stronger CNN Baseline for fair comparison with CapsNet
    Uses same conv backbone + better classifier + temporal modeling
    """
    def __init__(self, num_classes=6, input_size=(224, 224), use_attention=True):
        super().__init__()
        self.num_classes = num_classes
        self.use_attention = use_attention

        # Same convolutional feature extractor as CapsNet
        self.conv = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),

            nn.Conv2d(64, 128, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )

        # Add attention mechanism (simple channel attention like CapsNet has)
        self.channel_attention = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(256, 32, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 256, kernel_size=1),
            nn.Sigmoid()
        )

        # Spatial attention
        self.spatial_attention = nn.Sequential(
            nn.Conv2d(256, 32, kernel_size=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, kernel_size=1),
            nn.Sigmoid()
        )

        # Improved classifier with more capacity
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(256, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        """
        Args:
            x: Input tensor (B, T, C, H, W) - T frames per video
        """
        B, T, C, H, W = x.shape

        # Process each frame through conv layers with attention
        frame_features = []
        for t in range(T):
            feat = self.conv(x[:, t])

            # Apply attention (CNN + Multi-Scale SAM row) or skip it (Baseline CNN row)
            if self.use_attention:
                channel_att = self.channel_attention(feat)
                spatial_att = self.spatial_attention(feat)
                feat_attended = feat * channel_att * spatial_att
            else:
                feat_attended = feat
            frame_features.append(feat_attended)

        # Stack and aggregate temporally
        frame_stack = torch.stack(frame_features, dim=1)  # (B, T, C, H, W)

        # Better temporal aggregation: use both mean and max
        x_mean = frame_stack.mean(dim=1)
        x_max = frame_stack.max(dim=1)[0]
        x_agg = x_mean + 0.5 * x_max  # Combine mean and max pooling

        # Classify
        output = self.classifier(x_agg)

        return output

In [ ]:
def train_epoch_cnn(model, loader, optimizer, device, scheduler=None, clip_grad=1.0):
    """Training function for CNN"""
    model.train()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for x, y in tqdm(loader, desc="Training CNN"):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)

        loss = criterion(out, y)
        loss.backward()

        # Gradient clipping (important for stability)
        if clip_grad:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_grad)

        optimizer.step()
        if scheduler:
            scheduler.step()

        total_loss += loss.item()
        preds = out.argmax(dim=1)
        total_correct += (preds == y).sum().item()
        total_samples += y.size(0)

    avg_loss = total_loss / len(loader)
    accuracy = total_correct / total_samples * 100
    return avg_loss, accuracy

In [ ]:
def evaluate_model_cnn(model, data_loader, device, dataset_name="Dataset"):
    """Evaluation for CNN"""
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x, y in tqdm(data_loader, desc=f"Evaluating {dataset_name}"):
            x, y = x.to(device), y.to(device)
            output = model(x)
            preds = output.argmax(dim=-1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    accuracy = (np.array(all_preds) == np.array(all_labels)).mean() * 100
    print(f"{dataset_name} Accuracy: {accuracy:.2f}% ({len(all_preds)} videos)")

    return accuracy, all_preds, all_labels

## 4C. Sabour-style capsule baseline
A standalone capsule network with dynamic routing, no normalisation layers and no attention. This class is used by the cross-entropy routine; the margin-loss sweep (Section 19) defines `PureCapsNetFixed`, which creates the routing layer in `__init__` so that it is registered with the optimiser.

In [ ]:
class _SabourPrimaryCaps(nn.Module):
    def __init__(self, in_channels, out_capsules=32, out_dim=8, k=7, stride=2):
        super().__init__()
        self.out_capsules, self.out_dim = out_capsules, out_dim
        self.caps_conv = nn.Conv2d(in_channels, out_capsules * out_dim, k,
                                   stride=stride, padding=0)
    def squash(self, v, eps=1e-8):
        n = v.norm(dim=-1, keepdim=True)
        return (n**2 / (1 + n**2)) * v / (n + eps)
    def forward(self, x):
        B = x.size(0)
        x = self.caps_conv(x)
        H, W = x.shape[2], x.shape[3]
        x = x.view(B, self.out_capsules, self.out_dim, H, W)
        x = x.permute(0, 3, 4, 1, 2).contiguous().view(B, -1, self.out_dim)
        return self.squash(x)

class _SabourDigitCaps(nn.Module):
    def __init__(self, in_capsules, in_dim=8, out_capsules=11, out_dim=16, routing_iters=3):
        super().__init__()
        self.out_capsules, self.out_dim, self.routing_iters = out_capsules, out_dim, routing_iters
        self.W = nn.Parameter(0.01 * torch.randn(1, in_capsules, out_capsules, out_dim, in_dim))
    def squash(self, v, eps=1e-8):
        n = v.norm(dim=-1, keepdim=True)
        return (n**2 / (1 + n**2)) * v / (n + eps)
    def forward(self, u):
        B = u.shape[0]
        W = self.W.expand(B, -1, -1, -1, -1)
        u_hat = torch.matmul(W, u.unsqueeze(2).unsqueeze(-1)).squeeze(-1)
        b = torch.zeros(B, u.shape[1], self.out_capsules, 1, device=u.device)
        for r in range(self.routing_iters):
            c = F.softmax(b, dim=2)
            s = (c * u_hat).sum(dim=1)
            v = self.squash(s)
            if r < self.routing_iters - 1:
                b = b + (u_hat * v.unsqueeze(1)).sum(-1, keepdim=True)
        return v

class PureCapsNet(nn.Module):
    """Canonical capsule network baseline (Sabour et al., 2017), video-adapted."""
    def __init__(self, num_classes=11, routing_iters=3):
        super().__init__()
        # Shallow conv stem (plain conv-ReLU with downsampling): 112 -> 14.
        self.stem = nn.Sequential(
            nn.Conv2d(3,   32, 9, stride=2, padding=4), nn.ReLU(True),   # 112 -> 56
            nn.Conv2d(32,  64, 5, stride=2, padding=2), nn.ReLU(True),   # 56  -> 28
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.ReLU(True),   # 28  -> 14
        )
        self.primary = _SabourPrimaryCaps(128, 32, 8, k=7, stride=2)     # 14 -> 4 (512 caps)
        self.num_classes   = num_classes
        self.routing_iters = routing_iters
        self.use_sam       = False   # baseline has no attention module
        self.digit = None       # created once the capsule count is known
        self.activity = None    # alias to digit, set on first forward
    def forward(self, x, return_attention=False):
        B, T, C, H, W = x.shape
        f = self.stem(x.view(B * T, C, H, W))
        caps = self.primary(f)
        caps = caps.view(B, T, caps.shape[1], caps.shape[2]).mean(1)
        if self.digit is None:
            self.digit = _SabourDigitCaps(caps.shape[1], 8, self.num_classes, 16,
                                          self.routing_iters).to(caps.device)
            self.activity = self.digit   # alias: routing hooks use `.activity`
        return self.digit(caps)

print("PureCapsNet (canonical Sabour-style capsule baseline) defined.")

## 5. Attention configurations
`full_sam` (SAM-CapsNet), `no_sam` (Sabour-style baseline) and the six partial-attention variants.

In [ ]:
# Ablation configurations — 8 entries in the order specified
ABLATION_CONFIGS = [
    ('full_sam',
     dict(use_channel=True, use_spatial=True, use_point_prompt=True,
          use_temporal=True, use_learnable_fusion=True),
     'SAM-CapsNet (Proposed)',
     'Full model: channel + spatial + point prompt + temporal + learnable fusion'),

    ('no_sam',
     None,
     'Pure CapsNet',
     'Capsule network only, no attention'),

    ('ch_only',
     dict(use_channel=True,  use_spatial=False, use_point_prompt=False,
          use_temporal=False, use_learnable_fusion=False),
     'Channel only',
     'Channel attention branch only'),

    ('tm_only',
     dict(use_channel=False, use_spatial=False, use_point_prompt=False,
          use_temporal=True,  use_learnable_fusion=False),
     'Temporal only',
     'Temporal motion attention branch only'),

    ('sp_only',
     dict(use_channel=False, use_spatial=True,  use_point_prompt=False,
          use_temporal=False, use_learnable_fusion=False),
     'Spatial only',
     'Spatial attention branch only (no point prompt)'),

    ('ch_tm',
     dict(use_channel=True,  use_spatial=False, use_point_prompt=False,
          use_temporal=True,  use_learnable_fusion=False),
     'Channel + Temporal',
     'Channel and temporal branches'),

    ('tm_sp',
     dict(use_channel=False, use_spatial=True,  use_point_prompt=False,
          use_temporal=True,  use_learnable_fusion=False),
     'Temporal + Spatial',
     'Temporal and spatial branches (no channel, no PP)'),

    ('ch_sp',
     dict(use_channel=True,  use_spatial=True,  use_point_prompt=False,
          use_temporal=False, use_learnable_fusion=False),
     'Channel + Spatial',
     'Channel and spatial branches (no temporal, no PP)'),
]

CONFIG_BY_KEY = {k: (k, cfg, lbl, desc) for k, cfg, lbl, desc in ABLATION_CONFIGS}
print("{} ablation configurations registered.".format(len(ABLATION_CONFIGS)))
for k, _, lbl, _ in ABLATION_CONFIGS:
    print("  {:<28} {}".format(k, lbl))

In [ ]:
# Full attention configuration, used by the placement study and the UCF-101 sections.
FULL_SAM_CFG = dict(use_channel=True, use_spatial=True, use_point_prompt=True,
                    use_temporal=True, use_learnable_fusion=True)

## 6. Training utilities
`run_one_seed()` is the cross-entropy routine. `evaluate()` is then redefined so that it accepts both capsule outputs `(B, classes, 16)` and CNN logits `(B, classes)`.

In [ ]:
SEEDS = [42, 123, 456, 789, 2024]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def train_one_epoch(model, loader, optimizer, device, grad_clip=5.0, grad_accum=4):
    """
    Gradient accumulation + mixed precision for memory efficiency.
    grad_accum=4 with batch_size=2 -> effective batch=8 without the RAM cost.
    Mixed precision (autocast) halves activation memory on CUDA GPUs.
    """
    model.train()
    criterion = nn.CrossEntropyLoss()
    total_loss, correct, n = 0.0, 0, 0
    scaler = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None
    optimizer.zero_grad()

    for step, (x, y) in enumerate(tqdm(loader, desc="  train", leave=False)):
        x, y = x.to(device), y.to(device)

        if scaler is not None:
            with torch.cuda.amp.autocast():
                logits = model(x).norm(dim=-1)
                loss   = criterion(logits, y) / grad_accum
            scaler.scale(loss).backward()
        else:
            logits = model(x).norm(dim=-1)
            loss   = criterion(logits, y) / grad_accum
            loss.backward()

        total_loss += loss.item() * grad_accum
        correct    += (logits.detach().argmax(1) == y).sum().item()
        n          += y.size(0)

        if (step + 1) % grad_accum == 0 or (step + 1) == len(loader):
            if scaler is not None:
                if grad_clip:
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                if grad_clip:
                    nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()
            optimizer.zero_grad()
            if device.type == 'cuda':
                torch.cuda.empty_cache()

    return total_loss / len(loader), correct / n * 100


def evaluate(model, loader, device):
    model.eval()
    preds, labels = [], []
    for x, y in tqdm(loader, desc="  eval", leave=False):
        p = model(x.to(device)).norm(dim=-1).argmax(1).cpu()
        preds.extend(p.numpy())
        labels.extend(y.numpy())
    return (np.array(preds) == np.array(labels)).mean() * 100, np.array(preds), np.array(labels)


def run_one_seed(dataset, num_classes, sam_config, seed, num_epochs,
                 batch_size, lr, device):
    set_seed(seed)
    train_idx, val_idx, test_idx = make_stratified_splits(dataset, seed=seed)

    # num_workers for parallel data loading
    nw = 1 if os.name != 'nt' else 0   # keep low for UCF-101 memory
    train_ld = DataLoader(Subset(dataset, train_idx), batch_size=batch_size,
                          shuffle=True,  num_workers=nw, pin_memory=True)
    val_ld   = DataLoader(Subset(dataset, val_idx),   batch_size=batch_size,
                          shuffle=False, num_workers=nw, pin_memory=True)
    test_ld  = DataLoader(Subset(dataset, test_idx),  batch_size=batch_size,
                          shuffle=False, num_workers=nw, pin_memory=True)

    # sam_config=None builds the canonical Pure CapsNet baseline;
    # any non-None dict builds SAM-CapsNet with that attention configuration.
    if sam_config is None:
        model = PureCapsNet(num_classes=num_classes).to(device)
    else:
        model = SAMCapsNet(num_classes=num_classes, sam_config=sam_config).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    def lr_lambda(ep):
        warmup = 3
        if ep < warmup:
            return (ep + 1) / warmup
        return 0.5 * (1 + math.cos(math.pi * (ep - warmup) / max(num_epochs - warmup, 1)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    best_val, best_state, train_accs, val_accs = 0.0, None, [], []

    for ep in range(num_epochs):
        tr_loss, tr_acc = train_one_epoch(model, train_ld, opt, device,
                                          grad_accum=GRAD_ACCUM)
        val_acc, _, _   = evaluate(model, val_ld, device)
        scheduler.step()
        train_accs.append(tr_acc)
        val_accs.append(val_acc)
        if val_acc > best_val:
            best_val   = val_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        gate_str = ""
        if model.use_sam and hasattr(model.attention, 'gate_values'):
            gv = model.attention.gate_values()
            if gv:
                gate_str = "  [" + " ".join(
                    "{}={:.3f}".format(k, v.item()) for k, v in gv.items()) + "]"
        print("    Ep {:02d}/{} | loss {:.4f} | train {:.1f}% | val {:.1f}%{}".format(
              ep + 1, num_epochs, tr_loss, tr_acc, val_acc, gate_str))

    model.load_state_dict(best_state)
    test_acc, preds, gt = evaluate(model, test_ld, device)
    return dict(test_acc=test_acc, best_val=best_val,
                train_accs=train_accs, val_accs=val_accs,
                preds=preds, gt=gt, model=model)


def plot_training_curves(train_accs, val_accs, title="", save_path=None):
    fig, ax = plt.subplots(figsize=(8, 4))
    ep = range(1, len(train_accs) + 1)
    ax.plot(ep, train_accs, "b-o", ms=3, label="Train")
    ax.plot(ep, val_accs,   "r-o", ms=3, label="Val")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy (%)")
    ax.set_title("Training curves" + ("  -  " + title if title else ""))
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.show(); plt.close()


def plot_confusion_matrix(gt, preds, class_names, title="", save_path=None):
    cm_arr = confusion_matrix(gt, preds)
    acc    = (np.array(preds) == np.array(gt)).mean() * 100
    sz     = max(8, len(class_names))
    fig, ax = plt.subplots(figsize=(sz, sz - 1))
    sns.heatmap(cm_arr, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, linewidths=0.5)
    ax.set_title("Confusion Matrix" + ("  -  " + title if title else "")
                 + "\nAcc = {:.2f}%".format(acc))
    ax.set_ylabel("True"); ax.set_xlabel("Predicted")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.show(); plt.close()


def plot_ablation_bar(results, save_path=None):
    labels = [r["label"]     for r in results.values()]
    means  = [r["mean_test"] for r in results.values()]
    stds   = [r["std_test"]  for r in results.values()]
    colors = ["#FF5722" if k == "full_sam" else "#2196F3" for k in results]
    fig, ax = plt.subplots(figsize=(11, max(5, len(labels) * 0.65)))
    y = np.arange(len(labels))
    bars = ax.barh(y, means, xerr=stds, color=colors, capsize=4,
                   height=0.6, edgecolor="white", linewidth=0.5)
    ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=11)
    ax.set_xlabel("Mean Test Accuracy (%)", fontsize=12)
    ax.set_title("SAM-CapsNet Component Ablation Study (UCF-11)",
                 fontsize=13, fontweight="bold")
    ax.grid(axis="x", alpha=0.3); ax.set_xlim(0, max(means) + 14)
    for bar, m, s in zip(bars, means, stds):
        ax.text(bar.get_width() + s + 0.5,
                bar.get_y() + bar.get_height() / 2,
                "{:.2f} +/- {:.2f}%".format(m, s), va="center", fontsize=9.5)
    ax.legend(handles=[mpatches.Patch(color="#2196F3", label="Ablation variant"),
                       mpatches.Patch(color="#FF5722", label="SAM-CapsNet (Proposed)")],
              loc="lower right", fontsize=10)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show(); plt.close()


def plot_component_heatmap(results, save_path=None):
    comp_cols  = ["Channel", "Spatial", "Point\nPrompt", "Temporal", "Learnable\nlam"]
    comp_flags = {
        "full_sam": [1,1,1,1,1], "no_sam":  [0,0,0,0,0],
        "ch_only":  [1,0,0,0,0], "tm_only": [0,0,0,1,0],
        "sp_only":  [0,1,0,0,0], "ch_tm":   [1,0,0,1,0],
        "tm_sp":    [0,1,0,1,0], "ch_sp":   [1,1,0,0,0],
    }
    keys   = [k for k in results if k in comp_flags]
    labels = [results[k]["label"] for k in keys]
    matrix = np.array([comp_flags[k] for k in keys], dtype=float)
    accs   = [results[k]["mean_test"] for k in keys]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, max(5, len(keys) * 0.7)),
                                    gridspec_kw={"width_ratios": [3, 1]})
    annot = [["Y" if v else "" for v in row] for row in matrix]
    sns.heatmap(matrix, ax=ax1, cmap="Blues", vmin=0, vmax=1,
                xticklabels=comp_cols, yticklabels=labels,
                linewidths=0.5, linecolor="white", cbar=False,
                annot=annot, fmt="", annot_kws={"size": 13})
    ax1.set_title("Component Presence", fontweight="bold"); ax1.tick_params(labelsize=9)
    clrs = ["#FF5722" if k == "full_sam" else "#2196F3" for k in keys]
    ax2.barh(np.arange(len(keys)), accs, color=clrs, height=0.6)
    ax2.set_yticks([]); ax2.set_xlabel("Test Acc (%)")
    ax2.set_title("Accuracy", fontweight="bold")
    ax2.set_xlim(0, max(accs) + 14); ax2.grid(axis="x", alpha=0.3)
    for i, a in enumerate(accs):
        ax2.text(a + 0.5, i, "{:.1f}%".format(a), va="center", fontsize=9.5)
    plt.suptitle("SAM-CapsNet: Component presence vs accuracy", fontweight="bold")
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show(); plt.close()


print("Training utilities defined (num_workers=2, pin_memory=True).")

In [ ]:
# Redefine evaluate() in the live kernel with the shape-aware fix.
# Capsule models output (B, classes, dim) -> score = vector norm;
# CNN baselines (ResNet-18, MobileNetV2) output (B, classes) logits directly.
def evaluate(model, loader, device):
    model.eval()
    preds, labels = [], []
    import numpy as np
    with torch.no_grad():
        for x, y in tqdm(loader, desc="  eval", leave=False):
            out = model(x.to(device))
            scores = out.norm(dim=-1) if out.dim() == 3 else out
            p = scores.argmax(1).cpu()
            preds.extend(p.numpy())
            labels.extend(y.numpy())
    return (np.array(preds) == np.array(labels)).mean() * 100, np.array(preds), np.array(labels)

print("evaluate() redefined with shape guard — baselines will now work.")

In [ ]:
def compute_vector_angles(vectors):
    """Compute angles of vectors in high-dimensional space"""
    angles = []
    for vec in vectors:
        if len(vec) >= 2:
            angle = math.atan2(vec[1], vec[0])
            angles.append(angle)
        else:
            angles.append(0)
    return np.array(angles)

In [ ]:
def compute_pca_2d(vectors):
    """Compute PCA to project high-dimensional vectors to 2D"""
    vectors_centered = vectors - np.mean(vectors, axis=0)
    cov_matrix = np.cov(vectors_centered.T)
    eigenvals, eigenvecs = np.linalg.eigh(cov_matrix)

    idx = np.argsort(eigenvals)[::-1]
    eigenvecs = eigenvecs[:, idx]

    vectors_2d = vectors_centered @ eigenvecs[:, :2]
    return vectors_2d

## 7. Configuration — edit here
Set both dataset paths. Leave the other settings as they are to match the paper.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — Edit the two dataset paths below, then run every cell in order.
# ══════════════════════════════════════════════════════════════════════════════

# ── EDIT THESE TWO PATHS ──────────────────────────────────────────────────────
DATASET_PATH   = 'UCF11_updated_mpg'   # UCF-11 (11 classes) — used by most cells
UCF101_PATH    = 'UCF-101'             # Full UCF-101 (101 classes) — Sections 24-27

# ── Run switches ──────────────────────────────────────────────────────────────
QUICK_TEST     = False         # True = 1 seed, 2 epochs everywhere: fast end-to-end check, numbers not meaningful
RUN_UCF101     = True          # False = stop before the UCF-101 sections (24-27)

# ── Fixed settings (match the paper) ──────────────────────────────────────────
SAVE_DIR       = 'results_final'
CACHE_DIR      = '.cache_ucf11'
CLASS_SUBSET   = None          # None = use all 11 UCF-11 classes
NUM_EPOCHS     = 15            # UCF-11 ablation epochs
RR_EPOCHS      = 15            # not used by the paper sections
BATCH_SIZE     = 4
GRAD_ACCUM     = 4
LR             = 1e-3
NUM_SEEDS      = 5
if QUICK_TEST:
    NUM_EPOCHS, RR_EPOCHS, NUM_SEEDS = 2, 2, 1
SEEDS          = [42, 123, 456, 789, 2024][:NUM_SEEDS]
SAMPLE_FPS     = 5.0
INPUT_SIZE     = (112, 112)    # model expects 112x112 input (fixed 7x7 feature map)
SEQ_LEN        = 16

# ── Aliases so cells written against the UCF-101 benchmark names also work ─────
DATASET_PATH_UCF101 = UCF101_PATH
BATCH_SIZE_11       = BATCH_SIZE
BATCH_SIZE_101      = 2         # smaller batch for 101-class memory

os.makedirs(SAVE_DIR, exist_ok=True)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print(f'UCF-11  path : {DATASET_PATH}')
print(f'UCF-101 path : {UCF101_PATH}')
print(f'Save dir     : {SAVE_DIR}')
print(f'Epochs       : {NUM_EPOCHS}')
print(f'Quick test   : {QUICK_TEST}   UCF-101 sections: {RUN_UCF101}')
print(f'Seeds        : {SEEDS}')

## 8. Load UCF-11 and write the split file lists — Supplementary Table S3
The class counts printed here are those of Supplementary Table S3. The second cell writes the train/validation/test file list of every seed to `results_final/splits/`.

In [ ]:
# Load UCF-11 dataset. First run builds a disk cache; later runs load from it.
dataset = UCF11VideoDataset(DATASET_PATH,
                            sequence_length=SEQ_LEN,
                            target_size=INPUT_SIZE,
                            sample_fps=SAMPLE_FPS,
                            augment=False,
                            cache_dir=CACHE_DIR,
                            class_subset=CLASS_SUBSET)

NUM_CLASSES = len(dataset.class_to_idx)
CLASS_NAMES = [dataset.idx_to_class[i] for i in range(NUM_CLASSES)]

# ── Aliases for cells that use the *_11 naming convention ──────────────────────
dataset_11      = dataset
NUM_CLASSES_11  = NUM_CLASSES
CLASS_NAMES_11  = CLASS_NAMES

print('Classes ({}): {}'.format(NUM_CLASSES, CLASS_NAMES))
print('Total videos :', len(dataset))

In [ ]:
# Train / validation / test file lists for every seed.
import csv

ALL_SEEDS = [42, 123, 456, 789, 2024]

def export_split_lists(ds, root, tag, seeds=ALL_SEEDS):
    out_dir = os.path.join(SAVE_DIR, 'splits')
    os.makedirs(out_dir, exist_ok=True)
    root = Path(root)
    for seed in seeds:
        tr, va, te = make_stratified_splits(ds, seed=seed)
        fn = os.path.join(out_dir, f'{tag}_seed{seed}.csv')
        with open(fn, 'w', newline='') as f:
            w = csv.writer(f)
            w.writerow(['split', 'class', 'video'])
            for name, idx in (('train', tr), ('val', va), ('test', te)):
                for i in idx:
                    path, label = ds.samples[i]
                    w.writerow([name, ds.idx_to_class[label], Path(path).relative_to(root).as_posix()])
        print(f'{fn}: {len(tr)} train / {len(va)} val / {len(te)} test')

export_split_lists(dataset, DATASET_PATH, 'ucf11')

## 9. SAM-CapsNet, seed 42, cross-entropy routine
This model is used for Figures 2–4 and the latency measurement.

In [ ]:
# Quick single run — full SAM-CapsNet, seed 42
# Use this to verify the pipeline before running the full ablation.
print("Running quick check: SAM-CapsNet (full), seed=42 ...")
_, full_cfg, _, _ = CONFIG_BY_KEY['full_sam']

quick_r = run_one_seed(dataset, NUM_CLASSES, full_cfg,
                       seed=42, num_epochs=NUM_EPOCHS,
                       batch_size=BATCH_SIZE, lr=LR, device=DEVICE)

print("")
print("Test accuracy : {:.2f}%".format(quick_r['test_acc']))
print("Best val acc  : {:.2f}%".format(quick_r['best_val']))

plot_training_curves(quick_r['train_accs'], quick_r['val_accs'],
                     title='SAM-CapsNet (quick)',
                     save_path=SAVE_DIR + '/quick_curves.png')
plot_confusion_matrix(quick_r['gt'], quick_r['preds'], CLASS_NAMES,
                      title='SAM-CapsNet (quick)',
                      save_path=SAVE_DIR + '/quick_confusion.png')

## 10. CNN baselines — Table 1 CNN rows
`BaselineCNN` without attention and with channel and spatial attention (seed 42, softmax cross-entropy).

In [ ]:
# Requires Cell 8 (dataset loaded) to have run.
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

def run_cnn_variant(use_attention, tag, seed=42):
    set_seed(seed)
    n = len(dataset)
    idx = list(range(n))
    labels = [dataset[i][1] for i in idx] if False else None  # labels via dataset
    tr, te = train_test_split(idx, test_size=0.30, random_state=seed)
    va, te = train_test_split(te, test_size=0.50, random_state=seed)
    tl = DataLoader(Subset(dataset, tr), batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
    vl = DataLoader(Subset(dataset, va), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    xl = DataLoader(Subset(dataset, te), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    model = BaselineCNN(num_classes=NUM_CLASSES, input_size=INPUT_SIZE,
                        use_attention=use_attention).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)
    best_va, best_state = 0.0, None
    for ep in range(NUM_EPOCHS):
        train_epoch_cnn(model, tl, opt, DEVICE)
        va_acc, _, _ = evaluate_model_cnn(model, vl, DEVICE, 'val')
        if va_acc > best_va:
            best_va, best_state = va_acc, {k: v.cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    te_acc, preds, gts = evaluate_model_cnn(model, xl, DEVICE, 'test')
    print(f'[{tag}] test accuracy = {te_acc:.2f}%')
    return dict(acc=te_acc, preds=preds, gt=gts, model=model)

cnn_results = {}
cnn_results['no_attn'] = run_cnn_variant(use_attention=False, tag='Baseline CNN (softmax)')
cnn_results['with_attn'] = run_cnn_variant(use_attention=True,  tag='CNN + Multi-Scale SAM')
print('\nTable 1 rows 1-2:')
print(f"  Baseline CNN (softmax) : {cnn_results['no_attn']['acc']:.2f}%")
print(f"  CNN + Multi-Scale SAM  : {cnn_results['with_attn']['acc']:.2f}%")

## 11. Eight configurations, cross-entropy routine
Not the source of Table 1 (see Section 19). The paper uses this run for two things: the accuracy of the Sabour-style baseline under cross-entropy (Section II-E) and the last-seed SAM-CapsNet shown in Figure 5(b).
**Runtime:** about 3–4 hours.

In [ ]:
# Full ablation + capsule concentration measurement.
from torch.utils.data import DataLoader, Subset

configs_to_run = [c[0] for c in ABLATION_CONFIGS]
all_results  = {}
_mag_results = {'pure_caps': [], 'sam_caps': []}
_pure_model  = None   # saved for the viewpoint robustness cell
_sam_model   = None   # saved for the viewpoint robustness cell

for config_key in configs_to_run:
    _, sam_cfg, label, desc = CONFIG_BY_KEY[config_key]
    print(f'\n{"="*65}')
    print(f'CONFIG: {label}')
    print(f'  {desc}')
    print(f'{"="*65}')

    seed_test, all_preds, all_gt = [], [], []

    for s_idx, seed in enumerate(SEEDS):
        print(f'  Seed {seed} ({s_idx+1}/{len(SEEDS)}):')
        _, _, test_idx = make_stratified_splits(dataset, seed=seed)
        r = run_one_seed(dataset, NUM_CLASSES, sam_cfg, seed,
                         NUM_EPOCHS, BATCH_SIZE, LR, DEVICE)
        seed_test.append(r['test_acc'])
        all_preds.extend(r['preds'].tolist())
        all_gt.extend(r['gt'].tolist())
        print(f'  -> Seed {seed}: test = {r["test_acc"]:.2f}%')

        # ── Corrected capsule concentration (free — model in memory) ─────────
        if config_key in ('no_sam', 'full_sam'):
            _test_ld = DataLoader(Subset(dataset, test_idx),
                                  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
            _ha = {}
            def _fha(m, i, o): _ha['v'] = o.norm(dim=-1).detach().cpu()  # (B, N_classes)

            _h2 = r['model'].activity.register_forward_hook(_fha)
            _win_all, _sup_all, _lb_all = [], [], []
            r['model'].eval()
            with torch.no_grad():
                for _x, _y in _test_ld:
                    _ = r['model'](_x.to(DEVICE))
                    _am   = _ha['v']                              # (B, N_classes)
                    _pred = _am.argmax(dim=-1)                    # (B,)
                    _win  = _am[torch.arange(len(_pred)), _pred]  # (B,) winning mag
                    # Mean of ALL OTHER capsules per clip
                    _mask_out = torch.ones_like(_am, dtype=torch.bool)
                    _mask_out[torch.arange(len(_pred)), _pred] = False
                    _sup = _am[_mask_out].view(len(_pred), NUM_CLASSES - 1).mean(dim=-1)
                    _win_all.append(_win.numpy())
                    _sup_all.append(_sup.numpy())
                    _lb_all.append(_y.numpy())
            _h2.remove()

            _win_arr = np.concatenate(_win_all)   # (N_clips,)
            _sup_arr = np.concatenate(_sup_all)   # (N_clips,)
            _lb_arr  = np.concatenate(_lb_all)    # (N_clips,)
            _ratio   = float((_win_arr / (_sup_arr + 1e-8)).mean())
            _att_pct = float((_win_arr.mean() - _sup_arr.mean()) / _win_arr.mean() * 100)

            _pc_win, _pc_sup = {}, {}
            for _ci, _cn in enumerate(CLASS_NAMES):
                _m = _lb_arr == _ci
                if _m.sum():
                    _pc_win[_cn] = {'mean': float(_win_arr[_m].mean()),
                                    'std':  float(_win_arr[_m].std())}
                    _pc_sup[_cn] = {'mean': float(_sup_arr[_m].mean()),
                                    'std':  float(_sup_arr[_m].std())}

            _mkey = 'pure_caps' if config_key == 'no_sam' else 'sam_caps'
            _mag_results[_mkey].append({
                'seed': seed, 'n_test_clips': int(len(test_idx)),
                'win_mean':   float(_win_arr.mean()), 'win_std':  float(_win_arr.std()),
                'sup_mean':   float(_sup_arr.mean()), 'sup_std':  float(_sup_arr.std()),
                'concentration_ratio': _ratio,
                'attenuation_pct':     _att_pct,
                'per_class_win': _pc_win, 'per_class_sup': _pc_sup,
            })
            print(f'    [caps] win={_win_arr.mean():.3f}  sup={_sup_arr.mean():.3f}'
                  f'  ratio={_ratio:.2f}  att={_att_pct:.1f}%')

        # Save last-seed models for the viewpoint robustness cell
        if s_idx == len(SEEDS) - 1:
            if config_key == 'no_sam':
                _pure_model = r['model']
                _pure_test_idx = test_idx
            if config_key == 'full_sam':
                _sam_model  = r['model']
                _sam_test_idx = test_idx

            plot_training_curves(r['train_accs'], r['val_accs'], title=label,
                                 save_path=f'{SAVE_DIR}/{config_key}_curves.png')
            plot_confusion_matrix(r['gt'], r['preds'], CLASS_NAMES, title=label,
                                  save_path=f'{SAVE_DIR}/{config_key}_confusion.png')

    mean_t = float(np.mean(seed_test))
    std_t  = float(np.std(seed_test))
    all_preds_arr = np.array(all_preds)
    all_gt_arr    = np.array(all_gt)
    per_class_acc = {}
    for ci, cn in enumerate(CLASS_NAMES):
        mask = all_gt_arr == ci
        per_class_acc[cn] = float((all_preds_arr[mask] == ci).mean() * 100) if mask.sum() else 0.0

    print(f'  Result: {mean_t:.2f} +/- {std_t:.2f}%')
    all_results[config_key] = dict(
        label=label, description=desc,
        test_accs_per_seed=seed_test,
        mean_test=mean_t, std_test=std_t,
        per_class_acc=per_class_acc)
    with open(f'{SAVE_DIR}/ablation_table.json', 'w') as f:
        json.dump(all_results, f, indent=2)

# ── Save corrected capsule concentration stats ────────────────────────────────
def _agg_mag(key):
    e    = _mag_results[key]
    rats = np.array([x['concentration_ratio'] for x in e])
    atts = np.array([x['attenuation_pct']     for x in e])
    return {
        'win_mean':  float(np.mean([x['win_mean'] for x in e])),
        'win_std':   float(np.std( [x['win_mean'] for x in e])),
        'sup_mean':  float(np.mean([x['sup_mean'] for x in e])),
        'sup_std':   float(np.std( [x['sup_mean'] for x in e])),
        'concentration_ratio_mean': float(rats.mean()),
        'concentration_ratio_std':  float(rats.std()),
        'concentration_ratio_min':  float(rats.min()),
        'concentration_ratio_max':  float(rats.max()),
        'attenuation_mean': float(atts.mean()),
        'attenuation_std':  float(atts.std()),
        'attenuation_min':  float(atts.min()),
        'attenuation_max':  float(atts.max()),
        'n_seeds': len(e), 'n_test_clips': e[0]['n_test_clips'],
    }

_agg_data    = {k: _agg_mag(k) for k in ['pure_caps', 'sam_caps']}
_mag_summary = {'aggregate': _agg_data, 'per_seed': _mag_results,
                'class_names': CLASS_NAMES}
with open(f'{SAVE_DIR}/attenuation_stats.json', 'w') as f:
    json.dump(_mag_summary, f, indent=2)
print('Saved: attenuation_stats.json')

# ── Summary table ─────────────────────────────────────────────────────────────
print(f'\n{"="*65}')
print(f'ABLATION RESULTS (UCF-11, {NUM_CLASSES} classes, {len(SEEDS)} seeds, {NUM_EPOCHS} epochs)')
print(f'{"="*65}')
print(f'{"Config":<30}  {"Mean":>6}  {"Std":>5}')
print('-'*45)
for key, res in all_results.items():
    mark = '  <- proposed' if key == 'full_sam' else ''
    print(f'{res["label"]:<30}  {res["mean_test"]:>6.2f}  {res["std_test"]:>5.2f}%{mark}')
print('='*65)

plot_ablation_bar(all_results,      save_path=f'{SAVE_DIR}/ablation_bar.png')
plot_component_heatmap(all_results, save_path=f'{SAVE_DIR}/component_heatmap.png')

## 12. Confusion matrices — Figure 5
(a) scalar CNN baseline from Section 10; (b) last-seed SAM-CapsNet from Section 11.

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns, numpy as np, torch
from sklearn.metrics import confusion_matrix
from torch.utils.data import DataLoader, Subset

def _get_preds_gt(model, test_idx):
    """Run a model over its test split and collect predictions + ground truth."""
    model.eval()
    loader = DataLoader(Subset(dataset, test_idx), batch_size=BATCH_SIZE, shuffle=False)
    preds, gts = [], []
    with torch.no_grad():
        for xb, yb in loader:
            out = model(xb.to(DEVICE))
            # capsule output (B, C, 16) -> norm over last dim; CNN output (B, C) -> argmax
            p = out.norm(dim=-1).argmax(1) if out.dim() == 3 else out.argmax(1)
            preds.extend(p.cpu().tolist()); gts.extend(yb.tolist())
    return np.array(gts), np.array(preds)

def _cm_panel(ax, gt, preds, title):
    cm = confusion_matrix(gt, preds)
    acc = (preds == gt).mean() * 100
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax, cbar=False)
    ax.set_title(f'{title}\nAcc: {acc:.2f}%', fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted', fontsize=9); ax.set_ylabel('True', fontsize=9)
    ax.tick_params(labelsize=8); plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

# ── Guards: make the dependency explicit instead of a cryptic error ──────────
assert 'cnn_results' in dir(), (
    'Run the CNN baseline cell (Section 9B) first — it produces cnn_results '
    'for panel (a).')
assert '_sam_model' in dir() and _sam_model is not None, (
    'Run the Full Ablation cell (Section 10) first — it saves _sam_model '
    'for panel (b).')

# Panel (a): Baseline CNN — from cnn_results (Section 9B)
# Panel (b): SAM-CapsNet — stored by the ablation cell as _sam_model (last seed)
cnn_model = cnn_results['no_attn']['model']
cnn_gt    = np.array(cnn_results['no_attn']['gt'])
cnn_preds = np.array(cnn_results['no_attn']['preds'])

sam_gt, sam_preds = _get_preds_gt(_sam_model, _sam_test_idx)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
_cm_panel(ax1, cnn_gt,  cnn_preds,  '(a) Baseline CNN')
_cm_panel(ax2, sam_gt,  sam_preds,  '(b) SAM-CapsNet')
plt.suptitle('Figure 7. Confusion Matrix Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(SAVE_DIR + '/fig7_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: fig7_confusion_matrices.png')

## 13. Activity-capsule magnitudes and angles — Figure 2
One test clip, seed-42 SAM-CapsNet from Section 9.

In [ ]:
# Figure 2: capsule magnitudes and angles
# Requires seed-42 full_sam model from Cell 10

import matplotlib.pyplot as plt, matplotlib.ticker as ticker, numpy as np, torch

def plot_fig2(model, test_loader, device, class_names, save_dir):
    model.eval()
    x, y = next(iter(test_loader))
    with torch.no_grad():
        caps = model(x[:1].to(device))    # (1, num_classes, 16)
    vecs   = caps[0].cpu().numpy()
    mags   = np.linalg.norm(vecs, axis=1)
    angles = np.degrees([np.arctan2(v[1], v[0]) for v in vecs])
    colors = ['#E84040','#E84040','#4472C4','#375623','#7030A0',
              '#E87000','#FFC000','#833C00','#FF69B4','#808080','#A0A0A0']
    xp = np.arange(len(class_names))

    # (a) Magnitudes
    fig, ax = plt.subplots(figsize=(7, 5.5))
    bars = ax.bar(xp, mags, color=colors[:len(class_names)], edgecolor='white', linewidth=0.4, zorder=3)
    for bar, v in zip(bars, mags):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.006, f'{v:.3f}',
                ha='center', va='bottom', fontsize=9.5, fontweight='bold', color='#222')
    ax.set_title('Activity Capsule Magnitudes\n(Classification Confidence)', fontsize=13, fontweight='bold', pad=10)
    ax.set_ylabel('Magnitude', fontsize=11); ax.set_xticks(xp)
    ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=10)
    ax.set_ylim(0, max(mags)*1.15); ax.yaxis.set_major_locator(ticker.MultipleLocator(0.1))
    ax.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.5, zorder=0)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False); ax.set_axisbelow(True)
    fig.tight_layout()
    fig.savefig(save_dir+'/fig2a_magnitudes_300dpi.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

    # (b) Angles
    fig, ax = plt.subplots(figsize=(7, 5.5))
    bars = ax.bar(xp, angles, color=colors[:len(class_names)], edgecolor='white', linewidth=0.4, zorder=3)
    for bar, v in zip(bars, angles):
        off = 4 if v >= 0 else -8
        ax.text(bar.get_x()+bar.get_width()/2, v+off, f'{v:.1f}°',
                ha='center', va=('bottom' if v>=0 else 'top'),
                fontsize=9.5, fontweight='bold', color='#222')
    ax.set_title('Activity Capsule Angles\n(Feature Orientation)', fontsize=13, fontweight='bold', pad=10)
    ax.set_ylabel('Angle (degrees)', fontsize=11); ax.set_xticks(xp)
    ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=10)
    ax.axhline(0, color='black', linewidth=0.7, zorder=2)
    ax.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.5, zorder=0)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False); ax.set_axisbelow(True)
    fig.tight_layout()
    fig.savefig(save_dir+'/fig2b_angles_300dpi.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print('Saved fig2a and fig2b.')

from torch.utils.data import DataLoader, Subset

_, _, _te42 = make_stratified_splits(dataset, seed=42)
_fig_loader = DataLoader(Subset(dataset, _te42), batch_size=1, shuffle=False)
plot_fig2(quick_r['model'], _fig_loader, DEVICE, CLASS_NAMES, SAVE_DIR)

## 14. Interpretability — Figure 3 and Supplementary Figure S1
First cell: activity-capsule vectors of one clip (PCA projection, components, polar view). Second cell: spatial attention with the centre-of-mass trajectory.

In [ ]:
def visualize_activity_vectors(activity_output, model, dataset, save_plots=False):
    """
    COMPREHENSIVE activity capsule visualization with angles, magnitudes, and projections
    """
    vecs = activity_output.detach().cpu().numpy()
    magnitudes = np.linalg.norm(vecs, axis=1)
    angles = compute_vector_angles(vecs)

    # PCA projection
    if len(vecs) > 1:
        vecs_2d = compute_pca_2d(vecs)
    else:
        vecs_2d = vecs[:, :2] if vecs.shape[1] >= 2 else np.pad(vecs, ((0, 0), (0, 2-vecs.shape[1])), 'constant')

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))

    class_labels = [dataset.idx_to_class[i] for i in range(len(magnitudes))]
    colors = plt.cm.Set1(np.linspace(0, 1, len(magnitudes)))

    # 1. Magnitudes bar chart
    bars = axes[0, 0].bar(class_labels, magnitudes, color=colors)
    axes[0, 0].set_title('Activity Capsule Magnitudes\n(Classification Confidence)', fontweight='bold')
    axes[0, 0].set_ylabel('Magnitude')
    axes[0, 0].grid(axis='y', alpha=0.3)
    axes[0, 0].tick_params(axis='x', rotation=45)
    for bar, mag in zip(bars, magnitudes):
        axes[0, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                       f'{mag:.3f}', ha='center', va='bottom', fontsize=8)

    # 2. Angles bar chart
    axes[0, 1].bar(class_labels, np.degrees(angles), color=colors, alpha=0.7)
    axes[0, 1].set_title('Activity Capsule Angles\n(Feature Orientation)', fontweight='bold')
    axes[0, 1].set_ylabel('Angle (degrees)')
    axes[0, 1].grid(axis='y', alpha=0.3)
    axes[0, 1].tick_params(axis='x', rotation=45)
    for i, angle in enumerate(angles):
        axes[0, 1].text(i, np.degrees(angle) + 5, f'{np.degrees(angle):.1f}°',
                       ha='center', va='bottom', fontsize=8)

    # 3. Polar plot
    axes[0, 2] = plt.subplot(2, 3, 3, projection='polar')
    for i, (angle, magnitude) in enumerate(zip(angles, magnitudes)):
        axes[0, 2].arrow(0, 0, angle, magnitude, head_width=0.1, head_length=0.02,
                        fc=colors[i], ec=colors[i], alpha=0.8, linewidth=2)
        label_text = class_labels[i][:4]
        axes[0, 2].text(angle, magnitude + 0.05, label_text, ha='center', va='center',
                       bbox=dict(boxstyle="round,pad=0.1", facecolor=colors[i], alpha=0.7),
                       fontsize=8)
    axes[0, 2].set_title('Polar View: Angle + Magnitude', fontweight='bold')
    axes[0, 2].set_ylim(0, max(magnitudes) * 1.2)

    # 4. PCA 2D projection
    axes[1, 0].set_xlim(-2, 2)
    axes[1, 0].set_ylim(-2, 2)
    axes[1, 0].set_aspect('equal')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].set_title('PCA 2D Projection\n(Size = Magnitude)', fontweight='bold')
    for i, (vec_2d, magnitude) in enumerate(zip(vecs_2d, magnitudes)):
        scale_factor = magnitude * 3
        axes[1, 0].quiver(0, 0, vec_2d[0] * scale_factor, vec_2d[1] * scale_factor,
                         angles='xy', scale_units='xy', scale=1,
                         color=colors[i], alpha=0.8, width=0.008)
        label_text = class_labels[i][:6]
        axes[1, 0].text(vec_2d[0] * scale_factor * 1.2, vec_2d[1] * scale_factor * 1.2,
                       label_text, fontsize=9, ha='center',
                       bbox=dict(boxstyle="round,pad=0.2", facecolor=colors[i], alpha=0.7))
    axes[1, 0].set_xlabel('First Principal Component')
    axes[1, 0].set_ylabel('Second Principal Component')

    # 5. Heatmap of all components
    im = axes[1, 1].imshow(vecs, cmap='RdBu_r', aspect='auto')
    axes[1, 1].set_title('Activity Vector Components\n(All Dimensions)', fontweight='bold')
    axes[1, 1].set_xlabel('Component Index')
    axes[1, 1].set_ylabel('Activity Class')
    axes[1, 1].set_yticks(range(len(class_labels)))
    axes[1, 1].set_yticklabels([label[:8] for label in class_labels])
    plt.colorbar(im, ax=axes[1, 1])

    # 6. Angle vs Magnitude scatter
    axes[1, 2].scatter(np.degrees(angles), magnitudes, c=colors, s=150, alpha=0.8)
    for i, (angle, magnitude) in enumerate(zip(angles, magnitudes)):
        label_text = class_labels[i][:6]
        axes[1, 2].annotate(label_text, (np.degrees(angle), magnitude),
                           xytext=(5, 5), textcoords='offset points', fontsize=9)
    axes[1, 2].set_xlabel('Angle (degrees)')
    axes[1, 2].set_ylabel('Magnitude')
    axes[1, 2].set_title('Angle vs Magnitude\n(Activity Properties)', fontweight='bold')
    axes[1, 2].grid(True, alpha=0.3)

    plt.tight_layout()
    if save_plots:
        plt.savefig('activity_vectors_enhanced.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Print analysis
    print("\n" + "="*70)
    print("ACTIVITY CAPSULE ANALYSIS - ANGLES & MAGNITUDES")
    print("="*70)
    for i in range(len(magnitudes)):
        print(f"{class_labels[i]}:")
        print(f"  Magnitude (Confidence): {magnitudes[i]:.4f}")
        print(f"  Angle (Orientation): {np.degrees(angles[i]):.2f}°")
        print(f"  Vector norm: {np.linalg.norm(vecs[i]):.4f}")
    predicted = np.argmax(magnitudes)
    print(f"\nPredicted: {class_labels[predicted]} (Confidence: {magnitudes[predicted]:.4f})")

_x, _y = next(iter(_fig_loader))
quick_r['model'].eval()
with torch.no_grad():
    _caps = quick_r['model'](_x.to(DEVICE))[0]
visualize_activity_vectors(_caps, quick_r['model'], dataset)

In [ ]:
def visualize_attention_com(model, frames, idx_to_class, device, cmap='jet'):
    model.eval()
    if frames.dim() == 4:
        frames = frames.unsqueeze(0)
    frames = frames.to(device)
    B, T, C, H, W = frames.shape

    with torch.no_grad():
        out, att = model(frames, return_attention=True)
    spatial = att['spatial']

    vecs = out[0].cpu().numpy()
    mags = np.linalg.norm(vecs, axis=1)
    pred = idx_to_class[int(np.argmax(mags))]

    show   = np.linspace(0, T - 1, min(8, T), dtype=int)
    n      = len(show)
    cmap_fn = __import__('matplotlib').cm.get_cmap(cmap)
    trail  = []

    fig, axes = plt.subplots(3, n, figsize=(3 * n, 9))
    if n == 1:
        axes = axes.reshape(-1, 1)

    for col, fi in enumerate(show):
        frame = frames[0, fi].cpu().permute(1, 2, 0).numpy()
        frame = np.clip(frame * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406], 0, 1)
        am    = spatial[0, fi, 0].cpu().numpy()
        am    = cv2.resize(am, (W, H), interpolation=cv2.INTER_CUBIC)
        am    = (am - am.min()) / (am.max() - am.min() + 1e-8)

        yg, xg = np.meshgrid(np.arange(H), np.arange(W), indexing='ij')
        s  = am.sum() + 1e-8
        cx = int((am * xg).sum() / s)
        cy = int((am * yg).sum() / s)
        trail.append((cx, cy))

        att_rgb = cmap_fn(am)[:, :, :3]
        overlay = frame * 0.4 + att_rgb * 0.6

        for row, img in enumerate([frame, att_rgb, overlay]):
            axes[row, col].imshow(np.clip(img, 0, 1))
            axes[row, col].add_patch(plt.Circle((cx, cy), 8, color='red', zorder=5))
            if row == 2 and col > 0:
                px, py = trail[col - 1]
                axes[row, col].plot([px, cx], [py, cy], 'lime', lw=2, ls='--', zorder=4)
            axes[row, col].axis('off')
        axes[0, col].set_title("Frame {}".format(fi + 1), fontsize=8)

    for row, lbl in enumerate(['Original', 'Attention', 'Overlay']):
        axes[row, 0].set_ylabel(lbl, fontsize=9)
    plt.suptitle("Attention + CoM  |  Predicted: {}  (conf={:.3f})".format(
                 pred, mags.max()), fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    plt.close()


if 'quick_r' in dir() and quick_r['model'] is not None:
    train_idx, val_idx, test_idx = make_stratified_splits(dataset, seed=42)
    sample_idx  = test_idx[0]
    frames_vis, true_lbl = dataset[sample_idx]
    print("Visualising: {} (sample {})".format(
          dataset.idx_to_class[true_lbl], sample_idx))
    visualize_attention_com(quick_r['model'], frames_vis,
                            dataset.idx_to_class, DEVICE)
else:
    print("Run the quick check cell first, then re-run this cell.")

## 15. Per-class capsule-vector distributions — Figure 4
All clips (training, validation and test) of six classes, seed-42 SAM-CapsNet from Section 9. Each point is the true-class activity capsule of one clip: length against orientation angle in the plane of the first two principal components.

In [ ]:
from sklearn.decomposition import PCA
from torch.utils.data import DataLoader, Subset

FIG4_CLASSES = ['basketball', 'biking', 'diving', 'golf_swing', 'horse_riding', 'soccer_juggling']

def circular_mean_std_deg(angles_rad):
    """Circular mean and standard deviation in degrees (mean resultant length)."""
    cos_m, sin_m = np.cos(angles_rad).mean(), np.sin(angles_rad).mean()
    R = np.hypot(cos_m, sin_m)
    mean_angle = np.degrees(np.arctan2(sin_m, cos_m))
    std_angle  = np.degrees(np.sqrt(max(-2.0 * np.log(max(R, 1e-8)), 0.0)))
    return mean_angle, std_angle

def collect_class_capsule_vectors(model, ds, device, wanted):
    """True-class activity-capsule vector of every clip in the wanted classes."""
    model.eval()
    wanted_idx = {ci for ci, cn in ds.idx_to_class.items() if cn.lower() in wanted}
    idxs = [i for i, (_, lbl) in enumerate(ds.samples) if lbl in wanted_idx]
    loader = DataLoader(Subset(ds, idxs), batch_size=4, shuffle=False, num_workers=0)
    vecs = {ci: [] for ci in wanted_idx}
    with torch.no_grad():
        for frames, labels in loader:
            out = model(frames.to(device))                 # (B, classes, 16)
            for i in range(frames.size(0)):
                ci = int(labels[i].item())
                vecs[ci].append(out[i, ci].float().cpu().numpy())
    return vecs

vecs_by_class = collect_class_capsule_vectors(quick_r['model'], dataset, DEVICE, FIG4_CLASSES)
all_vecs = np.concatenate([np.stack(v) for v in vecs_by_class.values() if len(v) > 0])
pca4 = PCA(n_components=2).fit(all_vecs)

fig, axes = plt.subplots(2, 3, figsize=(15, 10), subplot_kw={'projection': 'polar'})
fig4_rows = []
for ax, ci in zip(axes.flat, sorted(vecs_by_class.keys())):
    vecs = np.stack(vecs_by_class[ci])
    proj = pca4.transform(vecs)
    angles = np.arctan2(proj[:, 1], proj[:, 0])
    mags = np.linalg.norm(vecs, axis=1)
    mean_ang, std_ang = circular_mean_std_deg(angles)
    ax.scatter(angles, mags, s=12, alpha=0.6)
    cname = dataset.idx_to_class[ci]
    ax.set_title(f'{cname} (n={len(vecs)})', fontsize=10)
    fig4_rows.append(dict(class_name=cname, n=int(len(vecs)), mean_angle_deg=float(mean_ang),
                          angle_std_deg=float(std_ang), mean_magnitude=float(mags.mean()),
                          magnitude_std=float(mags.std())))
plt.tight_layout()
plt.savefig(SAVE_DIR + '/figure4_interclass_capsule_analysis.png', dpi=300, bbox_inches='tight')
plt.show(); plt.close()

print('{:<18}{:>6}{:>20}{:>20}'.format('Class', 'n', 'Angle (deg)', 'Length'))
for r in fig4_rows:
    print('{:<18}{:>6}{:>12.1f} +/- {:<5.1f}{:>10.3f} +/- {:<5.3f}'.format(
        r['class_name'], r['n'], r['mean_angle_deg'], r['angle_std_deg'],
        r['mean_magnitude'], r['magnitude_std']))
print('Total clips:', sum(r['n'] for r in fig4_rows))
with open(SAVE_DIR + '/figure4_summary.json', 'w') as f:
    json.dump(fig4_rows, f, indent=2)

## 16. Capsule orientation test — Table 5
Cosine similarity and length ratio of the predicted-class capsule under horizontal flip and 90° clockwise rotation (55 test clips, seed 42, cross-entropy routine). The baseline column is not used in the paper.

In [ ]:
# Capsule Orientation Equivariance Test
# Standard protocol: measure cosine similarity change + magnitude stability.
# Trains both models fresh (seed 42) for a clean equivariance measurement.
# Zero retraining.

import numpy as np, torch, json, os, matplotlib.pyplot as plt
import torchvision.transforms.functional as TF
from torch.utils.data import DataLoader, Subset
from scipy.spatial.distance import cosine as cosine_dist

EQUIV_SAVE = os.path.join(SAVE_DIR, 'equivariance_test.json')

# ── Train SAM-CapsNet and Pure CapsNet at seed 42 for this test ──────────
# Train one seed if not already done
FULL_CFG = dict(use_channel=True, use_spatial=True, use_point_prompt=True,
                use_temporal=True, use_learnable_fusion=True)
print('Training SAM-CapsNet seed 42 on UCF-11 for equivariance test...')
set_seed(42)
r_equiv = run_one_seed(dataset_11, NUM_CLASSES_11, FULL_CFG, 42,
                       NUM_EPOCHS, BATCH_SIZE_11, LR, DEVICE)
equiv_model = r_equiv['model']
equiv_model.eval()
print(f'Model trained. Test acc: {r_equiv["test_acc"]:.2f}%')

# Also train Pure CapsNet for comparison
print('Training Pure CapsNet seed 42 on UCF-11...')
set_seed(42)
r_pure = run_one_seed(dataset_11, NUM_CLASSES_11, None, 42,
                      NUM_EPOCHS, BATCH_SIZE_11, LR, DEVICE)
pure_model = r_pure['model']
pure_model.eval()
print(f'Pure CapsNet test acc: {r_pure["test_acc"]:.2f}%')

# ── Helpers ───────────────────────────────────────────────────────────────────
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def apply_transform(frames, name):
    out = []
    for f in (frames * _STD + _MEAN).clamp(0, 1):
        if   name == 'hflip': f = TF.hflip(f)
        elif name == 'rot90': f = TF.rotate(f, 90)
        out.append((f - _MEAN) / _STD)
    return torch.stack(out)

def get_capsule_vector(model, frames, device):
    """Returns (predicted_class, 16-dim capsule vector, magnitude)."""
    with torch.no_grad():
        caps = model(frames.unsqueeze(0).to(device))  # (1, C, 16)
    vecs = caps[0].cpu().numpy()                       # (C, 16)
    mags = np.linalg.norm(vecs, axis=-1)               # (C,)
    pred = int(mags.argmax())
    return pred, vecs[pred], float(mags[pred])

# ── Select 5 clips per class from seed-42 test split ─────────────────────────
set_seed(42)
_, _, test_idx = make_stratified_splits(dataset_11, seed=42)
by_cls = {}
for i in test_idx:
    _, lbl = dataset_11.samples[i]; by_cls.setdefault(lbl, []).append(i)
rng = np.random.default_rng(42)
selected = []
for ci in range(NUM_CLASSES_11):
    pool   = by_cls.get(ci, [])
    picked = rng.choice(pool, size=min(5, len(pool)), replace=False)
    selected.extend((int(i), ci) for i in picked)
print(f'\nSelected {len(selected)} clips (5 per class x {NUM_CLASSES_11} classes)')

# ── Run equivariance test ─────────────────────────────────────────────────────
TRANSFORMS = [('hflip', 'Horizontal flip'), ('rot90', '90 deg CW rotation')]

results = {mname: {tn: {'cos_sim': [], 'mag_ratio': []}
                   for tn, _ in TRANSFORMS}
           for mname in ['sam_caps', 'pure_caps']}
records = []

for clip_i, (ds_idx, true_cls) in enumerate(selected):
    frames, _ = dataset_11[ds_idx]
    rec = {'true_class': CLASS_NAMES_11[true_cls], 'transforms': {}}

    for mname, model in [('sam_caps', equiv_model), ('pure_caps', pure_model)]:
        pred_orig, v_orig, mag_orig = get_capsule_vector(model, frames, DEVICE)

        for tn, tlabel in TRANSFORMS:
            frames_t = apply_transform(frames, tn)
            pred_t, v_t, mag_t = get_capsule_vector(model, frames_t, DEVICE)

            # Cosine similarity between original and transformed vectors
            # High similarity = NOT equivariant (orientation didn't change)
            # Low similarity  = equivariant (orientation DID change)
            cos_sim   = float(1 - cosine_dist(v_orig, v_t))
            mag_ratio = float(mag_t / (mag_orig + 1e-8))

            results[mname][tn]['cos_sim'].append(cos_sim)
            results[mname][tn]['mag_ratio'].append(mag_ratio)

            if mname == 'sam_caps':
                rec['transforms'][tn] = {
                    'cos_sim': cos_sim, 'mag_ratio': mag_ratio,
                    'pred_orig': pred_orig, 'pred_transformed': pred_t,
                }
    records.append(rec)

# ── Print equivariance table ───────────────────────────────────────────────────
print()
print('='*72)
print('CAPSULE ORIENTATION EQUIVARIANCE TEST (Section III-G)')
print('Protocol: Sabour et al. (2017) -- orientation changes, existence preserved')
print('='*72)
print()
print('Cosine similarity between original and transformed capsule vectors:')
print('  Lower cosine sim = orientation DID change = equivariant behaviour')
print('  (For reference: random 16-dim vectors have cosine sim ~0.0)')
print()
print(f'{"Condition":<22}  {"SAM-CapsNet":>22}  {"Pure CapsNet":>22}')
print(f'{"":22}  {"cos_sim (lower=better)":>22}  {"cos_sim":>22}')
print('-'*70)

equiv_rows = []
for tn, tlabel in TRANSFORMS:
    sam_cos  = np.array(results['sam_caps'][tn]['cos_sim'])
    pure_cos = np.array(results['pure_caps'][tn]['cos_sim'])
    sam_mag  = np.array(results['sam_caps'][tn]['mag_ratio'])
    pure_mag = np.array(results['pure_caps'][tn]['mag_ratio'])

    print(f'{tlabel:<22}  {sam_cos.mean():>8.3f} +/- {sam_cos.std():.3f}  '
          f'  {pure_cos.mean():>8.3f} +/- {pure_cos.std():.3f}')
    print(f'  Magnitude ratio         {sam_mag.mean():>8.3f} +/- {sam_mag.std():.3f}  '
          f'  {pure_mag.mean():>8.3f} +/- {pure_mag.std():.3f}')
    print()

    equiv_rows.append({
        'transform': tlabel,
        'sam_cos_mean': float(sam_cos.mean()), 'sam_cos_std': float(sam_cos.std()),
        'sam_mag_mean': float(sam_mag.mean()), 'sam_mag_std': float(sam_mag.std()),
        'pure_cos_mean': float(pure_cos.mean()), 'pure_cos_std': float(pure_cos.std()),
        'pure_mag_mean': float(pure_mag.mean()), 'pure_mag_std': float(pure_mag.std()),
        'n_clips': len(sam_cos),
    })

print('='*72)
print(f'N = {len(selected)} clips  |  5 per class  |  UCF-11 seed-42 test split')
print()
print('INTERPRETATION:')
print('  If cos_sim << 1.0 -> capsule orientation changes under transformation')
print('     -> equivariant behaviour (consistent with encoding viewpoint)')
print('  If mag_ratio ~1.0 -> capsule existence (magnitude) is stable')
print('     -> invariant existence (class identity preserved despite transform)')
print('  SAM-CapsNet vs Pure CapsNet: compare whether attention improves equivariance')

# ── Bar chart: cosine similarity comparison ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
transform_labels = [r['transform'] for r in equiv_rows]
x = np.arange(len(equiv_rows)); w = 0.35

for ax, metric, ylabel, title, ideal_note in [
    (axes[0], 'cos', 'Cosine Similarity (lower = more equivariant)',
     'Orientation Equivariance', 'Lower is better'),
    (axes[1], 'mag', 'Magnitude Ratio (closer to 1.0 = more invariant)',
     'Existence Invariance', 'Closer to 1.0 is better'),
]:
    sam_vals  = [r[f'sam_{metric}_mean']  for r in equiv_rows]
    sam_errs  = [r[f'sam_{metric}_std']   for r in equiv_rows]
    pure_vals = [r[f'pure_{metric}_mean'] for r in equiv_rows]
    pure_errs = [r[f'pure_{metric}_std']  for r in equiv_rows]

    ax.bar(x - w/2, pure_vals, w, yerr=pure_errs, capsize=4,
           label='Pure CapsNet', color='#90CAF9', edgecolor='#1976D2')
    ax.bar(x + w/2, sam_vals,  w, yerr=sam_errs,  capsize=4,
           label='SAM-CapsNet', color='#FF7043', edgecolor='#BF360C')

    ax.set_xticks(x); ax.set_xticklabels(transform_labels, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(f'{title}\n({ideal_note})', fontweight='bold')
    ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

    if metric == 'mag':
        ax.axhline(1.0, color='green', linestyle='--', alpha=0.6, label='Ideal (1.0)')
        ax.legend(fontsize=9)

plt.suptitle('Capsule Equivariance Test: Orientation Change vs Existence Stability\n'
             f'(n={len(selected)} clips, UCF-11, standard Sabour et al. protocol)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
fig_path = os.path.join(SAVE_DIR, 'equivariance_test.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show(); plt.close()

with open(EQUIV_SAVE, 'w') as f:
    json.dump({'rows': equiv_rows, 'records': records,
               'n_clips': len(selected), 'class_names': CLASS_NAMES_11}, f, indent=2)
print(f'\nSaved: {EQUIV_SAVE}  |  {fig_path}')

## 17. Inference latency — Section III-F
Per-clip forward-pass time of the seed-42 SAM-CapsNet, batch size 1.

In [ ]:
import time

def benchmark_latency(model, device, seq_len, input_size,
                       num_warmup=10, num_runs=100, batch_size=1):
    """
    Per-clip inference latency, forward pass only.
    Uses random dummy input at the model's real input shape -- this measures
    architecture cost, not video decode/IO/preprocessing time, consistent with
    how inference latency is usually reported in the literature. CUDA calls are
    synchronised before/after each timed pass so GPU work is fully captured.
    """
    model.eval()
    H, W = input_size
    dummy = torch.randn(batch_size, seq_len, 3, H, W, device=device)

    with torch.no_grad():
        for _ in range(num_warmup):
            _ = model(dummy)
        if device.type == 'cuda':
            torch.cuda.synchronize()

        times_ms = []
        for _ in range(num_runs):
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            _ = model(dummy)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t1 = time.perf_counter()
            times_ms.append((t1 - t0) * 1000.0)

    times_ms = np.array(times_ms)
    mean_ms, std_ms = float(times_ms.mean()), float(times_ms.std())
    clips_per_sec = 1000.0 / mean_ms * batch_size

    print("Latency benchmark: {} timed passes, batch={}, device={}".format(
        num_runs, batch_size, device))
    print("  Per-clip latency : {:.2f} +/- {:.2f} ms".format(mean_ms, std_ms))
    print("  Throughput       : {:.1f} clips/s".format(clips_per_sec))

    return dict(mean_ms=mean_ms, std_ms=std_ms, clips_per_sec=clips_per_sec,
                num_warmup=num_warmup, num_runs=num_runs, batch_size=batch_size,
                seq_len=seq_len, input_size=list(input_size), device=str(device))


# Prefer a model already trained earlier in the notebook; fall back to the
# quick-run model if the full experiments have not been run yet.
_bench_model = None
if 'proposed_ucf101_results' in dir() and proposed_ucf101_results.get('model') is not None:
    _bench_model = proposed_ucf101_results['model']
    _bench_tag   = 'proposed_ucf101'
elif 'quick_r' in dir():
    _bench_model = quick_r['model']
    _bench_tag   = 'quick_run'
else:
    raise RuntimeError("No trained model in memory -- run Cell 9 or Cell 9B first.")

latency_summary = benchmark_latency(_bench_model, DEVICE, SEQ_LEN, INPUT_SIZE,
                                     num_warmup=10, num_runs=100, batch_size=1)
latency_summary['tag'] = _bench_tag
latency_summary['num_classes'] = NUM_CLASSES

out_path = SAVE_DIR + '/latency_benchmark_{}.json'.format(_bench_tag)
with open(out_path, 'w') as f:
    json.dump(latency_summary, f, indent=2)
print("Saved:", out_path)

## 18. Attention-placement study — Section III-A
The same attention module before capsule formation, after capsule formation and inside the routing loop, with paired t-tests. Cross-entropy routine.

In [ ]:
# ============================================================================
# FIX B — R3.5 attention-placement ablation: 5 seeds, faithful "during-routing"
# variant (attention modulates the votes INSIDE the routing loop), + paired t-tests.
# Reports honestly whatever the data shows.
# ============================================================================
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, math, json
from torch.utils.data import DataLoader, Subset
from scipy import stats

# --- "after capsule formation" variant (attention on the primary-capsule grid) ---
class SAMCapsNet_After(SAMCapsNet):
    def __init__(self, num_classes=11, sam_config=None):
        super().__init__(num_classes, sam_config)
        if self.use_sam:
            ch = self.PRIM_CAPS * self.PRIM_DIM
            cfg = dict(FULL_SAM_CFG); cfg.update(sam_config or {})
            self.attention = MultiScaleSAMAttention(in_channels=ch, **cfg)
    def forward(self, x, return_attention=False):
        B,T,C,H,W = x.shape
        feats = self.conv(x.view(B*T,C,H,W)).view(B,T,256,self.FEAT_H,self.FEAT_W)
        grids=[]
        for t in range(T):
            pc = self.primary.caps_conv(feats[:,t])
            if self.use_sam: pc,_,_ = self.attention(pc, None, return_maps=False)
            grids.append(pc)
        grid = torch.stack(grids,1).mean(1)
        v = grid.view(B,self.PRIM_CAPS,self.PRIM_DIM,self.FEAT_H,self.FEAT_W)
        v = v.permute(0,3,4,1,2).contiguous().view(B,-1,self.PRIM_DIM)
        n=v.norm(dim=-1,keepdim=True); caps=(n**2/(1+n**2))*v/(n+1e-8)
        return self.activity(caps)

# --- "during routing" variant: attention weights the votes inside each routing iter ---
class ActivityCaps_AttnRouting(nn.Module):
    def __init__(self, in_capsules, in_dim=8, out_capsules=11, out_dim=16, routing_iters=3):
        super().__init__()
        self.out_capsules, self.out_dim, self.routing_iters = out_capsules, out_dim, routing_iters
        self.W = nn.Parameter(0.01*torch.randn(1,in_capsules,out_capsules,out_dim,in_dim))
        # attention over input capsules, applied within routing (comparable capacity to SAM gate)
        self.vote_attn = nn.Sequential(nn.Linear(out_dim, out_dim//2), nn.ReLU(True),
                                       nn.Linear(out_dim//2, 1))
    def squash(self, v, eps=1e-8):
        n=v.norm(dim=-1,keepdim=True); return (n**2/(1+n**2))*v/(n+eps)
    def forward(self, u):
        B=u.shape[0]; W=self.W.expand(B,-1,-1,-1,-1)
        u_hat = torch.matmul(W, u.unsqueeze(2).unsqueeze(-1)).squeeze(-1)   # (B,in,out,dim)
        # attention weight per (input-capsule, output-capsule) vote, from the vote vector
        a = torch.sigmoid(self.vote_attn(u_hat)).squeeze(-1).unsqueeze(-1)  # (B,in,out,1)
        u_hat = u_hat * a                                                    # modulate votes DURING routing
        b = torch.zeros(B, u.shape[1], self.out_capsules, 1, device=u.device)
        for r in range(self.routing_iters):
            c = F.softmax(b, dim=2)
            v = self.squash((c*u_hat).sum(1))
            if r < self.routing_iters-1:
                b = b + (u_hat*v.unsqueeze(1)).sum(-1, keepdim=True)
        return v

class SAMCapsNet_During(SAMCapsNet):
    def __init__(self, num_classes=11, sam_config=None):
        super().__init__(num_classes, sam_config)
        in_caps = self.PRIM_CAPS*self.FEAT_H*self.FEAT_W
        self.activity = ActivityCaps_AttnRouting(in_caps, self.PRIM_DIM, num_classes, 16)
        self.use_sam = False   # no attention on conv maps; attention lives in routing
    def forward(self, x, return_attention=False):
        B,T,C,H,W = x.shape
        feats = self.conv(x.view(B*T,C,H,W)).view(B,T,256,self.FEAT_H,self.FEAT_W)
        agg = self.pre_caps_norm(feats.mean(1))
        return self.activity(self.primary(agg))

def _build(key, nc):
    if key=='before': return SAMCapsNet(nc, sam_config=FULL_SAM_CFG)
    if key=='after':  return SAMCapsNet_After(nc, sam_config=FULL_SAM_CFG)
    if key=='during': return SAMCapsNet_During(nc, sam_config=FULL_SAM_CFG)

def _train_eval(key, seed):
    set_seed(seed)
    tr,va,te = make_stratified_splits(dataset, seed=seed)
    nw = 1 if os.name!='nt' else 0
    tl=DataLoader(Subset(dataset,tr),batch_size=BATCH_SIZE,shuffle=True,num_workers=nw,pin_memory=True)
    vl=DataLoader(Subset(dataset,va),batch_size=BATCH_SIZE,shuffle=False,num_workers=nw,pin_memory=True)
    xl=DataLoader(Subset(dataset,te),batch_size=BATCH_SIZE,shuffle=False,num_workers=nw,pin_memory=True)
    m=_build(key,NUM_CLASSES).to(DEVICE)
    opt=torch.optim.Adam(m.parameters(),lr=LR,weight_decay=1e-4)
    def ll(ep):
        w=3; return (ep+1)/w if ep<w else 0.5*(1+math.cos(math.pi*(ep-w)/max(NUM_EPOCHS-w,1)))
    sch=torch.optim.lr_scheduler.LambdaLR(opt,ll); best=0.0; bs=None
    for ep in range(NUM_EPOCHS):
        train_one_epoch(m,tl,opt,DEVICE,grad_accum=GRAD_ACCUM)
        va_acc,_,_=evaluate(m,vl,DEVICE); sch.step()
        if va_acc>best: best=va_acc; bs={k:v.cpu().clone() for k,v in m.state_dict().items()}
    m.load_state_dict(bs); te_acc,_,_=evaluate(m,xl,DEVICE); return te_acc

res={}
for key,label in [('before','Attention before capsule formation (proposed)'),
                  ('after','Attention after capsule formation'),
                  ('during','Attention during routing')]:
    accs=[_train_eval(key,s) for s in SEEDS]
    res[key]=accs
    print(f"{label}: {np.mean(accs):.2f} ± {np.std(accs):.2f}%  (seeds: {[f'{a:.1f}' for a in accs]})")

print("\n"+"="*72); print("R3.5 ATTENTION-PLACEMENT (UCF-11, 5 seeds, paired t-tests vs 'before')"); print("="*72)
before=np.array(res['before'])
for key,label in [('after','after capsule formation'),('during','during routing')]:
    arr=np.array(res[key]); t,p=stats.ttest_rel(before,arr)
    verdict = 'before significantly better' if (p<0.05 and before.mean()>arr.mean()) else \
              ('this variant significantly better' if (p<0.05 and arr.mean()>before.mean()) else 'no significant difference')
    print(f"before ({before.mean():.2f}) vs {label} ({arr.mean():.2f}): Δ={before.mean()-arr.mean():+.2f}, t={t:.2f}, p={p:.3f} → {verdict}")
json.dump(res, open(f'{SAVE_DIR}/placement_R3_5_5seed.json','w'), indent=2)

## 19. Margin-loss sweep — Table 1 (capsule rows) and Table 4
Eight configurations × five seeds: accuracy, macro precision/recall/F1, per-class accuracy and routing concentration. Per-seed accuracies are saved to `table1_marginloss_5seed.json`. The second cell checks that the configurations are distinct models.

In [ ]:
# ============================================================================
# STAGE 2 — Full Table 1 regeneration with MARGIN LOSS (8 configs x 5 seeds)
# All capsule models trained with the same correct margin loss, for a fair comparison.
# Computes accuracy, macro precision/recall/F1, per-class accuracy, and routing
# concentration in one sweep. Self-contained (redefines helpers).
# ============================================================================
import torch, torch.nn as nn, numpy as np, json, math, os
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import precision_recall_fscore_support

ABL = [
    ('full_sam', dict(use_channel=True,  use_spatial=True,  use_point_prompt=True,  use_temporal=True,  use_learnable_fusion=True),  'SAM-CapsNet (Proposed)'),
    ('no_sam',   None,                                                                                                               'Pure CapsNet'),
    ('ch_only',  dict(use_channel=True,  use_spatial=False, use_point_prompt=False, use_temporal=False, use_learnable_fusion=False), 'Channel only'),
    ('tm_only',  dict(use_channel=False, use_spatial=False, use_point_prompt=False, use_temporal=True,  use_learnable_fusion=False), 'Temporal only'),
    ('sp_only',  dict(use_channel=False, use_spatial=True,  use_point_prompt=False, use_temporal=False, use_learnable_fusion=False), 'Spatial only'),
    ('ch_tm',    dict(use_channel=True,  use_spatial=False, use_point_prompt=False, use_temporal=True,  use_learnable_fusion=True),  'Channel + Temporal'),
    ('tm_sp',    dict(use_channel=False, use_spatial=True,  use_point_prompt=False, use_temporal=True,  use_learnable_fusion=True),  'Temporal + Spatial'),
    ('ch_sp',    dict(use_channel=True,  use_spatial=True,  use_point_prompt=False, use_temporal=False, use_learnable_fusion=True),  'Channel + Spatial'),
]
CLASS_NAMES = [dataset.idx_to_class[i] for i in range(NUM_CLASSES)]
SAVE_JSON   = f'{SAVE_DIR}/table1_marginloss_5seed.json'

def margin_loss(norms, y, m_pos=0.9, m_neg=0.1, lam=0.5):
    T = torch.zeros_like(norms); T[torch.arange(len(y)), y] = 1.0
    return (T*torch.clamp(m_pos-norms,min=0.)**2 + lam*(1-T)*torch.clamp(norms-m_neg,min=0.)**2).sum(1).mean()

# Pure CapsNet with eager (trainable) digit layer
PURE_IN_CAPS = 32 * 4 * 4
class PureCapsNetFixed(nn.Module):
    def __init__(self, num_classes=11, routing_iters=3):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3,32,9,stride=2,padding=4), nn.ReLU(True),
            nn.Conv2d(32,64,5,stride=2,padding=2), nn.ReLU(True),
            nn.Conv2d(64,128,3,stride=2,padding=1), nn.ReLU(True))
        self.primary = _SabourPrimaryCaps(128,32,8,k=7,stride=2)
        self.digit   = _SabourDigitCaps(PURE_IN_CAPS,8,num_classes,16,routing_iters)
        self.activity = self.digit; self.use_sam=False
    def forward(self, x, return_attention=False):
        B,T,C,H,W = x.shape
        f = self.stem(x.view(B*T,C,H,W))
        caps = self.primary(f); caps = caps.view(B,T,caps.shape[1],caps.shape[2]).mean(1)
        return self.digit(caps)

def build(key, cfg):
    return PureCapsNetFixed(NUM_CLASSES) if key=='no_sam' else SAMCapsNet(NUM_CLASSES, sam_config=cfg)

def train_eval_full(key, cfg, seed):
    set_seed(seed)
    tr,va,te = make_stratified_splits(dataset, seed=seed)
    nw = 1 if os.name!='nt' else 0
    tl=DataLoader(Subset(dataset,tr),batch_size=BATCH_SIZE,shuffle=True,num_workers=nw,pin_memory=True)
    vl=DataLoader(Subset(dataset,va),batch_size=BATCH_SIZE,shuffle=False,num_workers=nw,pin_memory=True)
    xl=DataLoader(Subset(dataset,te),batch_size=BATCH_SIZE,shuffle=False,num_workers=nw,pin_memory=True)
    m = build(key,cfg).to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=LR, weight_decay=1e-4)
    def ll(ep):
        w=3; return (ep+1)/w if ep<w else 0.5*(1+math.cos(math.pi*(ep-w)/max(NUM_EPOCHS-w,1)))
    sch = torch.optim.lr_scheduler.LambdaLR(opt, ll)
    best, bstate = 0.0, None
    for ep in range(NUM_EPOCHS):
        m.train(); accum=0; opt.zero_grad()
        for x,y in tl:
            x,y=x.to(DEVICE),y.to(DEVICE)
            norms = m(x).norm(dim=-1)
            loss = margin_loss(norms,y)/GRAD_ACCUM
            loss.backward(); accum+=1
            if accum%GRAD_ACCUM==0:
                nn.utils.clip_grad_norm_(m.parameters(),5.0); opt.step(); opt.zero_grad()
        va_acc,_,_ = evaluate(m, vl, DEVICE); sch.step()
        if va_acc>best: best=va_acc; bstate={k:v.cpu().clone() for k,v in m.state_dict().items()}
    m.load_state_dict(bstate)
    te_acc, preds, labels = evaluate(m, xl, DEVICE)
    p,r,f1,_ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    # per-class acc + concentration
    per_cls = {CLASS_NAMES[ci]: float((preds[labels==ci]==ci).mean()*100) if (labels==ci).sum() else float('nan')
               for ci in range(NUM_CLASSES)}
    m.eval(); W_,S_,L_=[],[],[]
    with torch.no_grad():
        for x,y in xl:
            am=m(x.to(DEVICE)).norm(dim=-1); pr=am.argmax(1)
            win=am.max(1).values.cpu().numpy()
            msk=torch.ones_like(am,dtype=torch.bool); msk[torch.arange(len(pr)),pr]=False
            sup=am[msk].view(len(pr),NUM_CLASSES-1).mean(1).cpu().numpy()
            W_.append(win);S_.append(sup);L_.append(y.numpy())
    win=np.concatenate(W_);sup=np.concatenate(S_)
    conc=float(win.mean()/(sup.mean()+1e-8))
    del m
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return dict(acc=te_acc, prec=p, rec=r, f1=f1, per_class=per_cls, conc=conc)

results={}
for key,cfg,label in ABL:
    runs=[train_eval_full(key,cfg,s) for s in SEEDS]
    agg=lambda k:(float(np.mean([r[k] for r in runs])), float(np.std([r[k] for r in runs])))
    results[key]=dict(label=label,
                      acc=agg('acc'), prec=agg('prec'), rec=agg('rec'), f1=agg('f1'), conc=agg('conc'),
                      per_class={c:(float(np.mean([r['per_class'][c] for r in runs])),
                                    float(np.std([r['per_class'][c] for r in runs]))) for c in CLASS_NAMES},
                      acc_seeds=[r['acc'] for r in runs])
    a=results[key]['acc']; print(f"  {label:<26} acc {a[0]:.2f} ± {a[1]:.2f}%")
    json.dump(results, open(SAVE_JSON,'w'), indent=2)   # save after each config (resumable record)

print("\n"+"="*86)
print("TABLE 1 (margin loss, 5 seeds) — Classification performance on UCF-11")
print("="*86)
print(f"{'Configuration':<26}{'Accuracy':>16}{'Precision':>12}{'Recall':>10}{'F1':>10}{'Conc.':>9}")
print("-"*86)
for key,_,label in ABL:
    r=results[key]
    acc_cell = f"{r['acc'][0]:.2f}±{r['acc'][1]:.2f}"
    print(f"{label:<26}{acc_cell:>16}{r['prec'][0]:>12.4f}{r['rec'][0]:>10.4f}{r['f1'][0]:>10.4f}{r['conc'][0]:>8.1f}x")

In [ ]:
# Verify each attention config is a DIFFERENT model (param counts must differ)
for key, cfg, label in ABL:
    m = (PureCapsNetFixed(NUM_CLASSES) if key=='no_sam'
         else SAMCapsNet(NUM_CLASSES, sam_config=cfg))
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"  {label:<24} {n/1e6:6.3f}M params  cfg={cfg}")
    del m

## 20. Network with attention removed — Table 1 and Table 4
Identical encoder, normalisation and capsule layers to SAM-CapsNet, with the attention module switched off. The UCF-101 run of this network is not part of the paper and stays switched off (`QUICK_101 = True`).

In [ ]:
# ============================================================================
# FAIR PURE CAPSNET BASELINE — identical architecture to SAM-CapsNet, attention OFF.
# Replaces the confounded PureCapsNetFixed (which lacked BatchNorm/LayerNorm/width).
# Produces, for both UCF-11 and UCF-101, everything the 6 Pure-containing tables need:
#   accuracy, macro P/R/F1 (Table 1), per-class accuracy (Table 2),
#   per-seed + aggregate routing concentration (Tables 6/7, Supp S2),
#   equivariance under flip/rotation (Table 8).
# ============================================================================
import torch, torch.nn as nn, numpy as np, math, os, json
import torchvision.transforms.functional as TF
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import precision_recall_fscore_support

def _fair_pure(num_classes):
    return SAMCapsNet(num_classes, sam_config=None)   # SAME net, attention removed

@torch.no_grad()
def _metrics(model, loader):
    model.eval(); preds=[]; labels=[]; win=[]; sup=[]
    for x,y in loader:
        out = model(x.to(DEVICE))                     # (B, C, D)
        am = out.norm(dim=-1)                         # (B, C)
        p = am.argmax(1)
        preds.append(p.cpu().numpy()); labels.append(y.numpy())
        w = am.max(1).values.cpu().numpy()
        mask = torch.ones_like(am, dtype=torch.bool); mask[torch.arange(len(p)),p]=False
        s = am[mask].view(len(p), am.shape[1]-1).mean(1).cpu().numpy()
        win.append(w); sup.append(s)
    preds=np.concatenate(preds); labels=np.concatenate(labels)
    win=np.concatenate(win); sup=np.concatenate(sup)
    acc = 100*(preds==labels).mean()
    pr,rc,f1,_ = precision_recall_fscore_support(labels,preds,average='macro',zero_division=0)
    return acc, pr, rc, f1, preds, labels, win, sup

@torch.no_grad()
def _equivariance(model, loader, n_clips=55):
    """Cosine sim + magnitude ratio of predicted-class capsule under flip/rot90."""
    model.eval(); res={'hflip':{'cos':[],'mag':[]}, 'rot90':{'cos':[],'mag':[]}}
    seen=0
    for x,y in loader:
        x=x.to(DEVICE)
        o0=model(x); p=o0.norm(dim=-1).argmax(1)
        v0=o0[torch.arange(len(p)),p]
        for name in ['hflip','rot90']:
            xt=x.clone()
            B,T,C,H,W=xt.shape; xt=xt.view(B*T,C,H,W)
            xt = TF.hflip(xt) if name=='hflip' else TF.rotate(xt,90)
            xt=xt.view(B,T,C,H,W)
            ot=model(xt); vt=ot[torch.arange(len(p)),p]
            cos=torch.nn.functional.cosine_similarity(v0,vt,dim=-1)
            mag=(vt.norm(dim=-1)/(v0.norm(dim=-1)+1e-8))
            res[name]['cos']+=cos.cpu().tolist(); res[name]['mag']+=mag.cpu().tolist()
        seen+=len(p)
        if seen>=n_clips: break
    return {k:{'cos':(float(np.mean(v['cos'])),float(np.std(v['cos']))),
               'mag':(float(np.mean(v['mag'])),float(np.std(v['mag']))) } for k,v in res.items()}

def run_fair_pure(dataset, num_classes, class_names, bs, tag):
    print(f"\n########## FAIR PURE CAPSNET on {tag} ##########")
    per_seed_conc=[]; accs=[]; precs=[]; recs=[]; f1s=[]
    per_class_acc={c:[] for c in class_names}; per_class_conc={c:[] for c in class_names}
    equiv_seed0=None
    for si,seed in enumerate(SEEDS):
        set_seed(seed)
        tr,va,te = make_stratified_splits(dataset, seed=seed)
        nw=1 if os.name!='nt' else 0
        tl=DataLoader(Subset(dataset,tr),batch_size=bs,shuffle=True,num_workers=nw,pin_memory=True)
        vl=DataLoader(Subset(dataset,va),batch_size=bs,shuffle=False,num_workers=nw,pin_memory=True)
        xl=DataLoader(Subset(dataset,te),batch_size=bs,shuffle=False,num_workers=nw,pin_memory=True)
        m=_fair_pure(num_classes).to(DEVICE)
        opt=torch.optim.Adam(m.parameters(),lr=LR,weight_decay=1e-4)
        def ll(ep):
            w=3; return (ep+1)/w if ep<w else 0.5*(1+math.cos(math.pi*(ep-w)/max(NUM_EPOCHS-w,1)))
        sch=torch.optim.lr_scheduler.LambdaLR(opt,ll); best=0.0; bstate=None
        for ep in range(NUM_EPOCHS):
            m.train(); accum=0; opt.zero_grad()
            for x,y in tl:
                x,y=x.to(DEVICE),y.to(DEVICE)
                loss=margin_loss(m(x).norm(dim=-1),y)/GRAD_ACCUM
                loss.backward(); accum+=1
                if accum%GRAD_ACCUM==0:
                    nn.utils.clip_grad_norm_(m.parameters(),5.0); opt.step(); opt.zero_grad()
            va_acc,_,_=evaluate(m,vl,DEVICE); sch.step()
            if va_acc>best: best=va_acc; bstate={k:v.cpu().clone() for k,v in m.state_dict().items()}
        m.load_state_dict(bstate)
        acc,pr,rc,f1,preds,labels,win,sup = _metrics(m, xl)
        accs.append(acc); precs.append(pr); recs.append(rc); f1s.append(f1)
        per_seed_conc.append(float(win.mean()/(sup.mean()+1e-8)))
        for ci,cn in enumerate(class_names):
            mk=labels==ci
            if mk.sum():
                per_class_acc[cn].append(100*(preds[mk]==ci).mean())
                per_class_conc[cn].append(float(win[mk].mean()/(sup[mk].mean()+1e-8)))
        if si==0: equiv_seed0=_equivariance(m, xl)
        print(f"  seed {seed}: acc {acc:.2f}%  conc {per_seed_conc[-1]:.1f}x")
        del m
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    out=dict(tag=tag, acc_seeds=[float(a) for a in accs],
        accuracy=(float(np.mean(accs)),float(np.std(accs))),
        precision=float(np.mean(precs)), recall=float(np.mean(recs)), f1=float(np.mean(f1s)),
        per_seed_conc=per_seed_conc,
        overall_conc=float(np.mean(per_seed_conc)),
        per_class_acc={c:(float(np.mean(v)) if v else float('nan')) for c,v in per_class_acc.items()},
        per_class_conc={c:(float(np.mean(v)) if v else float('nan')) for c,v in per_class_conc.items()},
        equivariance=equiv_seed0)
    print(f"  => {tag}: acc {out['accuracy'][0]:.2f}±{out['accuracy'][1]:.2f}  "
          f"P {out['precision']:.4f} R {out['recall']:.4f} F1 {out['f1']:.4f}  conc {out['overall_conc']:.1f}x")
    json.dump(out, open(f'{SAVE_DIR}/fair_pure_{tag}.json','w'), indent=2)
    return out

# UCF-11 (the headline comparison)
CN11 = [dataset.idx_to_class[i] for i in range(NUM_CLASSES)]
fair_pure_11  = run_fair_pure(dataset,     NUM_CLASSES,     CN11, BATCH_SIZE,     'ucf11')

# UCF-101 (set QUICK_101=True first to smoke-test before the long run)
QUICK_101=True
if not QUICK_101:
    if 'dataset_101' not in globals():   # Section 24 has not run yet
        dataset_101 = UCF11VideoDataset(DATASET_PATH_UCF101, sequence_length=SEQ_LEN, target_size=INPUT_SIZE,
                                        sample_fps=SAMPLE_FPS, augment=False, cache_dir='.cache_ucf101_r1')
        NUM_CLASSES_101 = len(dataset_101.class_to_idx)
    CN101 = [dataset_101.idx_to_class[i] for i in range(NUM_CLASSES_101)]
    fair_pure_101 = run_fair_pure(dataset_101, NUM_CLASSES_101, CN101, BATCH_SIZE_101, 'ucf101')
else:
    print("\n[UCF-101 skipped: set QUICK_101=False to run it \u2014 it's the long one]")

## 21. Pre-trained ResNet-18 stem on UCF-11 — Table 3
SAM-CapsNet with its convolutional stem replaced by an ImageNet-pretrained ResNet-18 stem; capsule layers unchanged.

In [ ]:
# ============================================================================
# BACKBONE TRADE-OFF TEST — custom-stem SAM-CapsNet vs pretrained-ResNet18 stem
# Honest framing: quantify what the from-scratch choice costs/saves. Reports
# accuracy (5 seeds), parameter count, and inference time for BOTH. The pretrained
# variant is a COMPARISON POINT, not a replacement for the proposed model.
# ============================================================================
import torch, torch.nn as nn, numpy as np, math, os, time, copy
import torchvision
from torch.utils.data import DataLoader, Subset

FULL_SAM_CFG = dict(use_channel=True, use_spatial=True, use_point_prompt=True,
                    use_temporal=True, use_learnable_fusion=True)

class SAMCapsNet_ResNetStem(SAMCapsNet):
    """Identical to SAMCapsNet but replaces the custom conv stem with an
    ImageNet-pretrained ResNet-18 up to layer3 (256 channels), so everything
    after the stem (SAM attention, primary caps, routing) is unchanged."""
    def __init__(self, num_classes=11, sam_config=None, pretrained=True):
        super().__init__(num_classes=num_classes, sam_config=sam_config)
        w = torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        r = torchvision.models.resnet18(weights=w)
        # conv1..layer3 -> 256 channels at input/16 (14x14 for 224, 7x7 for 112)
        self.conv = nn.Sequential(r.conv1, r.bn1, r.relu, r.maxpool,
                                  r.layer1, r.layer2, r.layer3)
    # forward() inherited unchanged — stem output is still 256-channel feature maps

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

@torch.no_grad()
def inference_time(model, loader, n_batches=20):
    model.eval()
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t0 = time.time(); seen = 0
    for bi, (x, y) in enumerate(loader):
        _ = model(x.to(DEVICE)); seen += x.size(0)
        if bi + 1 >= n_batches: break
    if torch.cuda.is_available(): torch.cuda.synchronize()
    return (time.time() - t0) / max(seen, 1) * 1000   # ms per clip

def train_eval_backbone(build_fn, seed):
    set_seed(seed)
    tr, va, te = make_stratified_splits(dataset, seed=seed)
    nw = 1 if os.name != 'nt' else 0
    tl = DataLoader(Subset(dataset, tr), batch_size=BATCH_SIZE, shuffle=True,  num_workers=nw, pin_memory=True)
    vl = DataLoader(Subset(dataset, va), batch_size=BATCH_SIZE, shuffle=False, num_workers=nw, pin_memory=True)
    xl = DataLoader(Subset(dataset, te), batch_size=BATCH_SIZE, shuffle=False, num_workers=nw, pin_memory=True)
    m = build_fn().to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=LR, weight_decay=1e-4)
    def ll(ep):
        w=3; return (ep+1)/w if ep<w else 0.5*(1+math.cos(math.pi*(ep-w)/max(NUM_EPOCHS-w,1)))
    sch = torch.optim.lr_scheduler.LambdaLR(opt, ll)
    best, bstate = 0.0, None
    for ep in range(NUM_EPOCHS):
        m.train(); accum=0; opt.zero_grad()
        for x,y in tl:
            x,y=x.to(DEVICE),y.to(DEVICE)
            loss = margin_loss(m(x).norm(dim=-1), y)/GRAD_ACCUM
            loss.backward(); accum+=1
            if accum%GRAD_ACCUM==0:
                nn.utils.clip_grad_norm_(m.parameters(),5.0); opt.step(); opt.zero_grad()
        va,_,_ = evaluate(m, vl, DEVICE); sch.step()
        if va>best: best=va; bstate={k:v.cpu().clone() for k,v in m.state_dict().items()}
    m.load_state_dict(bstate)
    te_acc,_,_ = evaluate(m, xl, DEVICE)
    t_ms = inference_time(m, xl)
    params = count_params(m)
    del m
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return te_acc, params, t_ms

variants = [
    ('Custom stem (proposed)',      lambda: SAMCapsNet(NUM_CLASSES, sam_config=FULL_SAM_CFG)),
    ('ResNet-18 stem (pretrained)', lambda: SAMCapsNet_ResNetStem(NUM_CLASSES, sam_config=FULL_SAM_CFG, pretrained=True)),
]
rows = {}
for label, build in variants:
    accs, ts = [], []
    params = 0
    for seed in SEEDS:
        a, p, t = train_eval_backbone(build, seed)
        accs.append(a); ts.append(t); params = p
        print(f"  [{label}] seed {seed}: acc {a:.2f}%  {t:.1f} ms/clip")
    rows[label] = dict(acc=(float(np.mean(accs)), float(np.std(accs))),
                       params=params, ms=float(np.mean(ts)))

print("\n" + "="*78)
print("BACKBONE TRADE-OFF (UCF-11, 5 seeds, margin loss, same protocol)")
print("="*78)
print(f"{'Variant':<30}{'Accuracy':>16}{'Params':>12}{'ms/clip':>10}")
print("-"*78)
for label, r in rows.items():
    acc_cell = f"{r['acc'][0]:.2f}±{r['acc'][1]:.2f}"
    print(f"{label:<30}{acc_cell:>16}{r['params']/1e6:>10.2f}M{r['ms']:>10.1f}")
import json; json.dump(rows, open(f'{SAVE_DIR}/backbone_tradeoff.json','w'), indent=2, default=str)
print(f"\nSaved: {SAVE_DIR}/backbone_tradeoff.json")

## 22. Per-class routing concentration of the Sabour-style baseline — Supplementary Table S2

In [ ]:
# ============================================================================
# Pure CapsNet per-class concentration — measured on the WORKING model
# (margin loss + eager digit layer = the recipe of Section 19). Earlier concentration
# cells measured a broken Pure model (untrained routing / 9% collapse) -> 0.0x.
# This trains Pure correctly AND measures concentration on the SAME model.
# ============================================================================
import numpy as np, torch, torch.nn as nn, math, os, json
from torch.utils.data import DataLoader, Subset

CLASS_NAMES = [dataset.idx_to_class[i] for i in range(NUM_CLASSES)]

def _pure_train_and_conc(seed):
    set_seed(seed)
    tr, va, te = make_stratified_splits(dataset, seed=seed)
    nw = 1 if os.name != 'nt' else 0
    tl = DataLoader(Subset(dataset, tr), batch_size=BATCH_SIZE, shuffle=True,  num_workers=nw, pin_memory=True)
    xl = DataLoader(Subset(dataset, te), batch_size=BATCH_SIZE, shuffle=False, num_workers=nw, pin_memory=True)
    m = PureCapsNetFixed(NUM_CLASSES).to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=LR, weight_decay=1e-4)
    assert id(m.digit.W) in {id(p) for g in opt.param_groups for p in g['params']}, "digit.W not in optimizer"
    def ll(ep):
        w=3; return (ep+1)/w if ep<w else 0.5*(1+math.cos(math.pi*(ep-w)/max(NUM_EPOCHS-w,1)))
    sch = torch.optim.lr_scheduler.LambdaLR(opt, ll)
    for ep in range(NUM_EPOCHS):
        m.train(); accum=0; opt.zero_grad()
        for x,y in tl:
            x,y=x.to(DEVICE),y.to(DEVICE)
            loss = margin_loss(m(x).norm(dim=-1), y)/GRAD_ACCUM   # working recipe
            loss.backward(); accum+=1
            if accum%GRAD_ACCUM==0:
                nn.utils.clip_grad_norm_(m.parameters(),5.0); opt.step(); opt.zero_grad()
        sch.step()
    # accuracy sanity + concentration on the SAME trained model
    acc,_,_ = evaluate(m, xl, DEVICE)
    m.eval(); W_,S_,L_=[],[],[]
    with torch.no_grad():
        for x,y in xl:
            am = m(x.to(DEVICE)).norm(dim=-1); pr=am.argmax(1)
            win=am.max(1).values.cpu().numpy()
            msk=torch.ones_like(am,dtype=torch.bool); msk[torch.arange(len(pr)),pr]=False
            sup=am[msk].view(len(pr),NUM_CLASSES-1).mean(1).cpu().numpy()
            W_.append(win);S_.append(sup);L_.append(y.numpy())
    del m
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return acc, np.concatenate(W_), np.concatenate(S_), np.concatenate(L_)

pc={c:[] for c in CLASS_NAMES}; ov=[]; accs=[]
for seed in SEEDS:
    print(f"  Pure CapsNet (margin loss) seed {seed} ...")
    acc,w,s,lb = _pure_train_and_conc(seed); accs.append(acc)
    ov.append(float(w.mean()/(s.mean()+1e-8)))
    for ci,cn in enumerate(CLASS_NAMES):
        mk=lb==ci
        if mk.sum(): pc[cn].append(float(w[mk].mean()/(s[mk].mean()+1e-8)))

mean_acc = float(np.mean(accs))
print("\n" + "="*60)
print("Pure CapsNet concentration (margin loss, 5 seeds)")
print("="*60)
print(f"ACCURACY SANITY CHECK: {mean_acc:.2f}%")
if mean_acc < 50:
    print(">>> WARNING: accuracy collapsed (<50%). Model did NOT train — concentration INVALID. Do not use.")
else:
    print(">>> OK: model trained (~75% expected). Concentration numbers below are VALID.")
print(f"\nOverall concentration: {np.mean(ov):.1f}x")
for c in CLASS_NAMES:
    print(f"  {c:<20}{np.mean(pc[c]):>6.1f} ± {np.std(pc[c]):.1f}x")
json.dump({'accuracy':accs,'overall':ov,'per_class':{c:pc[c] for c in CLASS_NAMES}},
          open(f'{SAVE_DIR}/pure_concentration_MARGINLOSS.json','w'), indent=2)
print(f"\nSaved: {SAVE_DIR}/pure_concentration_MARGINLOSS.json")

## 23. Significance tests and per-class accuracy — Section III-A and Table 2
Paired t-tests over the same seeds (and therefore the same splits) for SAM-CapsNet against the network with attention removed and against every other configuration of Table 1. Also prints the per-class accuracies of the margin-loss runs.

In [ ]:
from scipy import stats

with open(f'{SAVE_DIR}/table1_marginloss_5seed.json') as f:
    _t1 = json.load(f)
with open(f'{SAVE_DIR}/fair_pure_ucf11.json') as f:
    _off = json.load(f)

_full = np.array(_t1['full_sam']['acc_seeds'], dtype=float)
_rows = [('Same network, attention removed', np.array(_off['acc_seeds'], dtype=float))]
_rows += [(v['label'], np.array(v['acc_seeds'], dtype=float)) for k, v in _t1.items() if k != 'full_sam']

def _ms(a):
    sd = a.std(ddof=1) if len(a) > 1 else float('nan')
    return f'{a.mean():6.2f} +/- {a.std():4.2f} (sample s.d. {sd:4.2f})'

print(f'Seeds: {SEEDS}')
print(f"{'SAM-CapsNet (proposed)':<34}{_ms(_full)}")
print('-' * 100)
print(f"{'Compared with':<34}{'accuracy, mean +/- population s.d.':<40}{'diff (pp)':>10}{'t':>8}{'p':>8}")
for name, arr in _rows:
    if len(_full) > 1:
        t, p = stats.ttest_rel(_full, arr)
        tail = f'{_full.mean() - arr.mean():>+10.2f}{t:>8.2f}{p:>8.3f}'
    else:
        tail = '   (paired test needs at least two seeds)'
    print(f'{name:<34}{_ms(arr):<40}{tail}')

print()
print('Per-class accuracy (%), margin loss, mean over seeds')
print(f"{'Class':<22}{'SAM-CapsNet':>14}{'Attention removed':>20}{'Sabour-style':>16}")
for c in _t1['full_sam']['per_class']:
    print(f"{c:<22}{_t1['full_sam']['per_class'][c][0]:>14.1f}{_off['per_class_acc'][c]:>20.1f}{_t1['no_sam']['per_class'][c][0]:>16.1f}")

## 24. Load UCF-101 and define the CNN baselines
ImageNet-pretrained ResNet-18 and MobileNetV2 with temporal mean pooling. The first run builds the UCF-101 frame cache, which takes several hours.

In [ ]:
if not RUN_UCF101:
    raise RuntimeError('Stopped before the UCF-101 sections because RUN_UCF101 = False (Section 7). '
                       'All UCF-11 sections above have finished.')

In [ ]:
# UCF-101 full benchmark
# Trains SAM-CapsNet, ResNet-18, MobileNetV2 under identical protocol.

import torchvision.models as tv_models
from torch.utils.data import DataLoader, Subset

UCF101_SAVE = os.path.join(SAVE_DIR, 'ucf101_benchmark.json')

# ── Load UCF-101 ──────────────────────────────────────────────────────────────
print('Loading UCF-101...')
dataset_101 = UCF11VideoDataset(
    DATASET_PATH_UCF101, sequence_length=SEQ_LEN, target_size=INPUT_SIZE,
    sample_fps=SAMPLE_FPS, augment=False, cache_dir='.cache_ucf101_r1',
    class_subset=None)
NUM_CLASSES_101 = len(dataset_101.class_to_idx)
CLASS_NAMES_101 = [dataset_101.idx_to_class[i] for i in range(NUM_CLASSES_101)]
print(f'UCF-101: {NUM_CLASSES_101} classes, {len(dataset_101)} videos')
print(f'First 5 classes: {CLASS_NAMES_101[:5]}')


# ── Baseline model builders ───────────────────────────────────────────────────
class ResNet18Baseline(nn.Module):
    """ResNet-18 fine-tuned for video: mean-pool frames then classify."""
    def __init__(self, num_classes):
        super().__init__()
        backbone = tv_models.resnet18(weights=tv_models.ResNet18_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*list(backbone.children())[:-1])  # up to avgpool
        self.classifier = nn.Linear(512, num_classes)

    def forward(self, x):
        # x: (B, T, C, H, W)
        B, T, C, H, W = x.shape
        x_flat = x.view(B * T, C, H, W)
        feats  = self.features(x_flat).squeeze(-1).squeeze(-1)   # (B*T, 512)
        feats  = feats.view(B, T, 512).mean(dim=1)               # (B, 512) temporal mean
        return self.classifier(feats)                             # (B, num_classes)


class MobileNetV2Baseline(nn.Module):
    """MobileNetV2 fine-tuned for video: mean-pool frames then classify."""
    def __init__(self, num_classes):
        super().__init__()
        backbone = tv_models.mobilenet_v2(
            weights=tv_models.MobileNet_V2_Weights.IMAGENET1K_V1)
        self.features   = backbone.features
        self.classifier = nn.Linear(1280, num_classes)

    def forward(self, x):
        B, T, C, H, W = x.shape
        x_flat = x.view(B * T, C, H, W)
        feats  = self.features(x_flat)
        feats  = feats.mean(dim=[-2, -1])        # global avg pool: (B*T, 1280)
        feats  = feats.view(B, T, 1280).mean(1)  # temporal mean: (B, 1280)
        return self.classifier(feats)


def run_baseline_seed(dataset, num_classes, model_class, seed, num_epochs,
                      batch_size, lr, device, grad_accum=1):
    """Train a baseline model (ResNet-18 or MobileNetV2) for one seed."""
    set_seed(seed)
    train_idx, val_idx, test_idx = make_stratified_splits(dataset, seed=seed)
    nw = 1 if os.name != 'nt' else 0
    train_ld = DataLoader(Subset(dataset, train_idx), batch_size=batch_size,
                          shuffle=True,  num_workers=nw, pin_memory=True)
    val_ld   = DataLoader(Subset(dataset, val_idx),   batch_size=batch_size,
                          shuffle=False, num_workers=nw, pin_memory=True)
    test_ld  = DataLoader(Subset(dataset, test_idx),  batch_size=batch_size,
                          shuffle=False, num_workers=nw, pin_memory=True)

    model = model_class(num_classes).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    lr_lambda = lambda ep: 0.5 ** (ep // 7)
    scheduler = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    best_val, best_state = 0.0, None
    train_accs, val_accs = [], []

    for ep in range(num_epochs):
        model.train()
        correct, total = 0, 0
        opt.zero_grad()
        scaler = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None
        for step, (x, y) in enumerate(train_ld):
            x, y = x.to(device), y.to(device)
            if scaler:
                with torch.cuda.amp.autocast():
                    logits = model(x)
                    loss   = criterion(logits, y) / grad_accum
                scaler.scale(loss).backward()
            else:
                logits = model(x)
                loss   = criterion(logits, y) / grad_accum
                loss.backward()
            correct += (logits.detach().argmax(1) == y).sum().item()
            total   += y.size(0)
            if (step + 1) % grad_accum == 0 or (step + 1) == len(train_ld):
                if scaler:
                    scaler.step(opt); scaler.update()
                else:
                    opt.step()
                opt.zero_grad()
        train_acc = correct / total * 100
        val_acc, _, _ = evaluate(model, val_ld, device)
        train_accs.append(train_acc); val_accs.append(val_acc)
        if val_acc > best_val:
            best_val  = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if (ep + 1) % 5 == 0:
            print(f'    Ep {ep+1}/{num_epochs}  train={train_acc:.1f}%  val={val_acc:.1f}%')
        scheduler.step()

    model.load_state_dict(best_state)
    test_acc, preds, gt = evaluate(model, test_ld, device)
    return dict(test_acc=test_acc, best_val=best_val, model=model,
                preds=preds, gt=gt, train_accs=train_accs, val_accs=val_accs)


# ── Counting parameters ───────────────────────────────────────────────────────
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6

In [ ]:
export_split_lists(dataset_101, DATASET_PATH_UCF101, 'ucf101')

## 25. UCF-101 with the shared optimiser settings (learning rate 1e-3) — Table 3 and Section III-F
SAM-CapsNet in the cross-entropy routine, then ResNet-18 and MobileNetV2 with softmax cross-entropy. Same splits, input size, optimiser settings and number of epochs.

In [ ]:
# ── Run all three models on UCF-101 ──────────────────────────────────────────
FULL_SAM_CFG = dict(use_channel=True, use_spatial=True, use_point_prompt=True,
                    use_temporal=True, use_learnable_fusion=True)

MODELS_101 = [
    ('sam_capsnet', 'SAM-CapsNet (Proposed)', 'capsnet'),
    ('resnet18',    'ResNet-18',              'resnet18'),
    ('mobilenetv2', 'MobileNetV2',            'mobilenetv2'),
]

ucf101_results = {}

for model_key, model_label, arch_key in MODELS_101:
    print(f'\n{"="*65}')
    print(f'MODEL: {model_label}  (UCF-101, {NUM_CLASSES_101} classes)')
    print(f'{"="*65}')

    seed_test = []
    for s_idx, seed in enumerate(SEEDS):
        print(f'  Seed {seed} ({s_idx+1}/{len(SEEDS)}):')
        if arch_key == 'capsnet':
            r = run_one_seed(dataset_101, NUM_CLASSES_101, FULL_SAM_CFG, seed,
                             NUM_EPOCHS, BATCH_SIZE_101, LR, DEVICE)
        elif arch_key == 'resnet18':
            r = run_baseline_seed(dataset_101, NUM_CLASSES_101, ResNet18Baseline,
                                  seed, NUM_EPOCHS, BATCH_SIZE_101, LR, DEVICE,
                                  grad_accum=GRAD_ACCUM)
        else:
            r = run_baseline_seed(dataset_101, NUM_CLASSES_101, MobileNetV2Baseline,
                                  seed, NUM_EPOCHS, BATCH_SIZE_101, LR, DEVICE,
                                  grad_accum=GRAD_ACCUM)

        seed_test.append(r['test_acc'])
        print(f'  -> Seed {seed}: test = {r["test_acc"]:.2f}%')

    mean_t = float(np.mean(seed_test))
    std_t  = float(np.std(seed_test))

    # Parameter count (from last seed model)
    n_params = count_params(r['model'])

    ucf101_results[model_key] = {
        'label':          model_label,
        'test_accs':      seed_test,
        'mean_test':      mean_t,
        'std_test':       std_t,
        'n_params_M':     n_params,
    }
    print(f'  Result: {mean_t:.2f} +/- {std_t:.2f}%  ({n_params:.2f}M params)')
    with open(UCF101_SAVE, 'w') as f:
        json.dump(ucf101_results, f, indent=2)

# ── Print Table 3 Section A ───────────────────────────────────────────────────
print()
print('='*70)
print(f'UCF-101, learning rate {LR:g} for all three models')
print('No pre-training beyond ImageNet; same split, epochs, frame count')
print('='*70)
print(f'{"Method":<28}  {"Accuracy":>12}  {"Params":>10}')
print('-'*55)
for mk, res in ucf101_results.items():
    print(f'{res["label"]:<28}  '
          f'{res["mean_test"]:>6.2f} +/- {res["std_test"]:.2f}%  '
          f'{res["n_params_M"]:>7.2f}M')
print('='*70)
print(f'Protocol: 70/15/15 stratified split, {NUM_EPOCHS} epochs, '
      f'{SEQ_LEN} frames/clip, 112x112, 5 seeds')
print(f'Saved: {UCF101_SAVE}')

## 26. ResNet-18 and MobileNetV2 with a fine-tuning learning rate (1e-4)
Same models, splits, input size, epochs, batch size and schedule as Section 25; only the learning rate changes. A learning rate of 1e-3 is high for fine-tuning ImageNet-pretrained networks, so this section gives the baselines a fairer setting. Results are saved after every seed, and an interrupted run resumes where it stopped.

In [ ]:
LR_FINETUNE    = 1e-4
UCF101_FT_SAVE = os.path.join(SAVE_DIR, 'ucf101_baselines_finetune_lr.json')

ucf101_ft_results = {}
if os.path.exists(UCF101_FT_SAVE):                       # resume after an interruption
    with open(UCF101_FT_SAVE) as f:
        ucf101_ft_results = json.load(f)

FT_MODELS = [('resnet18',    'ResNet-18 (ImageNet)',   ResNet18Baseline),
             ('mobilenetv2', 'MobileNetV2 (ImageNet)', MobileNetV2Baseline)]

for key, label, model_class in FT_MODELS:
    rec = ucf101_ft_results.get(key)
    if rec is None or rec.get('num_epochs') != NUM_EPOCHS or rec.get('lr') != LR_FINETUNE:
        rec = dict(label=label, lr=LR_FINETUNE, num_epochs=NUM_EPOCHS, test_accs={}, best_vals={})
        ucf101_ft_results[key] = rec
    print('=' * 65)
    print(f'{label}  (UCF-101, learning rate {LR_FINETUNE:g}, {NUM_EPOCHS} epochs)')
    print('=' * 65)
    for seed in SEEDS:
        if str(seed) in rec['test_accs']:
            print(f"  Seed {seed}: already done, test = {rec['test_accs'][str(seed)]:.2f}%")
            continue
        print(f'  Seed {seed}:')
        r = run_baseline_seed(dataset_101, NUM_CLASSES_101, model_class, seed,
                              NUM_EPOCHS, BATCH_SIZE_101, LR_FINETUNE, DEVICE,
                              grad_accum=GRAD_ACCUM)
        rec['test_accs'][str(seed)] = float(r['test_acc'])
        rec['best_vals'][str(seed)] = float(r['best_val'])
        rec['n_params_M'] = float(sum(p.numel() for p in r['model'].parameters() if p.requires_grad) / 1e6)
        print(f"  -> Seed {seed}: test = {r['test_acc']:.2f}%  (best val {r['best_val']:.2f}%)")
        del r
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        with open(UCF101_FT_SAVE, 'w') as f:
            json.dump(ucf101_ft_results, f, indent=2)
    accs = [rec['test_accs'][str(s)] for s in SEEDS]
    rec['mean_test'], rec['std_test'] = float(np.mean(accs)), float(np.std(accs))
    with open(UCF101_FT_SAVE, 'w') as f:
        json.dump(ucf101_ft_results, f, indent=2)

print()
print('=' * 72)
print(f'UCF-101 CNN baselines: effect of the learning rate ({len(SEEDS)} seeds, {NUM_EPOCHS} epochs)')
print('=' * 72)
print(f"{'Model':<26}{'lr = ' + format(LR, 'g'):>20}{'lr = ' + format(LR_FINETUNE, 'g'):>20}{'Params':>9}")
print('-' * 72)
for key, label, _ in FT_MODELS:
    old, new = ucf101_results[key], ucf101_ft_results[key]
    print(f"{label:<26}{old['mean_test']:>12.2f} +/- {old['std_test']:<5.2f}"
          f"{new['mean_test']:>12.2f} +/- {new['std_test']:<5.2f}{new['n_params_M']:>8.2f}M")
print('=' * 72)
print('Standard deviations are the population form (divisor n), as in the rest of the notebook.')
print(f'Saved: {UCF101_FT_SAVE}')

## 27. Capsule models on UCF-101 with the margin loss — Table 3 and Section III-F
SAM-CapsNet, the pre-trained-stem variant, the Sabour-style baseline and the single-branch variants, with paired t-tests. The baseline's routing weights are scaled by K/11 at initialisation; without this it stays at chance level on 101 classes (see the comment in the cell).

**Note on the archived run.** The executed copy of this notebook archived on Zenodo was stopped after the three models reported in the paper (Sabour-style baseline, SAM-CapsNet and the pre-trained-stem variant; five seeds each). The single-branch variants listed in `CONFIGS` were not completed and are not used in the paper, so the summary table and t-tests at the end of the cell were not printed. Section 27B computes them from the saved per-seed accuracies.

In [ ]:
# ============================================================================
# UCF-101 REPLICATION — Core + paired t-tests
# Answers: backbone better? params? speed? do attention configs separate on UCF-101?
# per-class accuracy per config. QUICK=True first to smoke-test before the full run.
# ============================================================================
import torch, torch.nn as nn, numpy as np, math, os, time, json
import torchvision
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import precision_recall_fscore_support
from scipy import stats

QUICK = QUICK_TEST                                  # follows Section 7 (True = 1 seed, 2 epochs)
U101_SEEDS  = SEEDS[:1] if QUICK else SEEDS
U101_EPOCHS = NUM_EPOCHS      # 2 under QUICK_TEST, 15 otherwise
SAVE_U101   = f'{SAVE_DIR}/ucf101_replication.json'

FULL_SAM_CFG = dict(use_channel=True, use_spatial=True, use_point_prompt=True, use_temporal=True, use_learnable_fusion=True)

class SAMCapsNet_ResNetStem(SAMCapsNet):
    def __init__(self, num_classes, sam_config=None, pretrained=True):
        super().__init__(num_classes=num_classes, sam_config=sam_config)
        w = torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        r = torchvision.models.resnet18(weights=w)
        self.conv = nn.Sequential(r.conv1, r.bn1, r.relu, r.maxpool, r.layer1, r.layer2, r.layer3)

# Sabour-style baseline for a large label space. Routing starts with uniform coupling 1/K and the squash
# function is quadratic near zero, so with the 11-class weight scale the class capsules start at length
# ~1e-8 for K = 101: the loss gradient is then far smaller than the weight-decay term and the model stays
# at chance (1.10%). Scaling the routing weights by K/11 restores the 11-class starting signal.
# Set PURE_W_SCALE_101 = 1.0 to reproduce the collapse.
PURE_W_SCALE_101 = NUM_CLASSES_101 / 11
def build_pure_101(nc):
    m = PureCapsNetFixed(nc)
    with torch.no_grad():
        m.digit.W.mul_(PURE_W_SCALE_101)
    return m

# Core (supervisor's Qs) + branch configs (needed so paired tests have something to compare)
CONFIGS = [
    ('pure',        'Pure CapsNet',                 build_pure_101),
    ('full_sam',    'SAM-CapsNet (custom stem)',    lambda nc: SAMCapsNet(nc, sam_config=FULL_SAM_CFG)),
    ('resnet_stem', 'SAM-CapsNet (ResNet-18 stem)', lambda nc: SAMCapsNet_ResNetStem(nc, sam_config=FULL_SAM_CFG, pretrained=True)),
    ('ch_only',     'Channel only',  lambda nc: SAMCapsNet(nc, sam_config=dict(use_channel=True,  use_spatial=False, use_point_prompt=False, use_temporal=False, use_learnable_fusion=False))),
    ('tm_only',     'Temporal only', lambda nc: SAMCapsNet(nc, sam_config=dict(use_channel=False, use_spatial=False, use_point_prompt=False, use_temporal=True,  use_learnable_fusion=False))),
    ('sp_only',     'Spatial only',  lambda nc: SAMCapsNet(nc, sam_config=dict(use_channel=False, use_spatial=True,  use_point_prompt=False, use_temporal=False, use_learnable_fusion=False))),
]

CLASS_NAMES_101 = [dataset_101.idx_to_class[i] for i in range(NUM_CLASSES_101)]
def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

@torch.no_grad()
def infer_ms(model, loader, n=15):
    model.eval()
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t0=time.time(); seen=0
    for bi,(x,y) in enumerate(loader):
        _=model(x.to(DEVICE)); seen+=x.size(0)
        if bi+1>=n: break
    if torch.cuda.is_available(): torch.cuda.synchronize()
    return (time.time()-t0)/max(seen,1)*1000

def train_eval_101(build, seed):
    set_seed(seed)
    tr,va,te = make_stratified_splits(dataset_101, seed=seed)
    nw = 1 if os.name!='nt' else 0; bs = BATCH_SIZE_101
    tl=DataLoader(Subset(dataset_101,tr),batch_size=bs,shuffle=True,num_workers=nw,pin_memory=True)
    vl=DataLoader(Subset(dataset_101,va),batch_size=bs,shuffle=False,num_workers=nw,pin_memory=True)
    xl=DataLoader(Subset(dataset_101,te),batch_size=bs,shuffle=False,num_workers=nw,pin_memory=True)
    m=build(NUM_CLASSES_101).to(DEVICE)
    opt=torch.optim.Adam(m.parameters(),lr=LR,weight_decay=1e-4)
    def ll(ep):
        w=3; return (ep+1)/w if ep<w else 0.5*(1+math.cos(math.pi*(ep-w)/max(U101_EPOCHS-w,1)))
    sch=torch.optim.lr_scheduler.LambdaLR(opt,ll); best=0.0; bstate=None
    for ep in range(U101_EPOCHS):
        m.train(); accum=0; opt.zero_grad()
        for x,y in tl:
            x,y=x.to(DEVICE),y.to(DEVICE)
            loss=margin_loss(m(x).norm(dim=-1),y)/GRAD_ACCUM
            loss.backward(); accum+=1
            if accum%GRAD_ACCUM==0:
                nn.utils.clip_grad_norm_(m.parameters(),5.0); opt.step(); opt.zero_grad()
        va,_,_=evaluate(m,vl,DEVICE); sch.step()
        if va>best: best=va; bstate={k:v.cpu().clone() for k,v in m.state_dict().items()}
    m.load_state_dict(bstate)
    acc,preds,labels=evaluate(m,xl,DEVICE)
    p,r,f1,_=precision_recall_fscore_support(labels,preds,average='macro',zero_division=0)
    per_cls={CLASS_NAMES_101[ci]: float((preds[labels==ci]==ci).mean()*100) if (labels==ci).sum() else float('nan')
             for ci in range(NUM_CLASSES_101)}
    params=count_params(m); ms=infer_ms(m,xl)
    del m
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return dict(acc=acc, prec=p, rec=r, f1=f1, params=params, ms=ms, per_class=per_cls)

results={}
for key,label,build in CONFIGS:
    runs=[train_eval_101(build,s) for s in U101_SEEDS]
    agg=lambda k:(float(np.mean([r[k] for r in runs])), float(np.std([r[k] for r in runs])))
    results[key]=dict(label=label, acc=agg('acc'), prec=agg('prec'), rec=agg('rec'), f1=agg('f1'),
                      params=runs[0]['params'], ms=float(np.mean([r['ms'] for r in runs])),
                      per_class={c:float(np.mean([r['per_class'][c] for r in runs])) for c in CLASS_NAMES_101},
                      acc_seeds=[r['acc'] for r in runs])
    a=results[key]['acc']; print(f"  {label:<30} acc {a[0]:.2f}±{a[1]:.2f}  {results[key]['params']/1e6:.2f}M  {results[key]['ms']:.1f}ms")
    json.dump(results, open(SAVE_U101,'w'), indent=2)

# ---- summary table ----
print("\n"+"="*80)
print(f"UCF-101 REPLICATION ({'QUICK' if QUICK else 'FULL'}, {len(U101_SEEDS)} seeds, {U101_EPOCHS} ep)")
print("="*80)
print(f"{'Config':<30}{'Accuracy':>16}{'MacroF1':>10}{'Params':>10}{'ms/clip':>9}")
print("-"*80)
for key,label,_ in CONFIGS:
    r=results[key]; acc_cell=f"{r['acc'][0]:.2f}±{r['acc'][1]:.2f}"
    print(f"{label:<30}{acc_cell:>16}{r['f1'][0]:>10.4f}{r['params']/1e6:>9.2f}M{r['ms']:>9.1f}")

# ---- paired t-tests: full SAM vs each other config (only meaningful with >1 seed) ----
if len(U101_SEEDS) > 1:
    print("\n"+"="*80); print("PAIRED t-TESTS: SAM-CapsNet (full) vs each config (UCF-101, same seeds)"); print("="*80)
    full_accs=np.array(results['full_sam']['acc_seeds'])
    for key,label,_ in CONFIGS:
        if key=='full_sam': continue
        arr=np.array(results[key]['acc_seeds'])
        t,p=stats.ttest_rel(full_accs,arr); d=full_accs.mean()-arr.mean()
        sig = 'significant' if p<0.05 else 'n.s.'
        print(f"  full vs {label:<28} Δ={d:+6.2f}pp  t={t:+6.2f}  p={p:.3f}  ({sig})")
else:
    print("\n[paired t-tests skipped: QUICK mode has 1 seed. Set QUICK=False for significance tests.]")
print(f"\nSaved: {SAVE_U101}")

## 27B. UCF-101 capsule models: summary and paired t-tests — Table 3 and Section III-F
Reads `results_final/ucf101_replication.json`, written by Section 27, and prints the mean, the standard deviations and the paired t-tests quoted in Section III-F. Standard deviations after ± are the population form, as in the rest of the notebook.

In [ ]:
# Summary and paired t-tests for the UCF-101 capsule models, from the per-seed accuracies
# that Section 27 saved to ucf101_replication.json. Needs no GPU and no dataset.
import json, numpy as np
from scipy import stats
_r = json.load(open('results_final/ucf101_replication.json'))
print(f"{'Model':<32}{'Accuracy (%)':>16}{'sample s.d.':>13}{'Macro F1':>10}{'Params':>9}")
print('-' * 80)
for _k, _v in _r.items():
    _a = np.array(_v['acc_seeds'])
    print(f"{_v['label']:<32}{_a.mean():>9.2f} ± {_a.std():.2f}{_a.std(ddof=1):>13.2f}{_v['f1'][0]:>10.4f}{_v['params']/1e6:>8.2f}M")
print('\nPer-seed test accuracy (seeds 42, 123, 456, 789, 2024)')
for _k, _v in _r.items():
    print(f"  {_v['label']:<32}" + '  '.join(f'{x:.2f}' for x in _v['acc_seeds']))
print('\nPaired t-tests against SAM-CapsNet (custom stem), same seeds and splits')
_full = np.array(_r['full_sam']['acc_seeds'])
for _k, _v in _r.items():
    if _k == 'full_sam': continue
    _a = np.array(_v['acc_seeds']); _t, _p = stats.ttest_rel(_a, _full)
    print(f"  {_v['label']:<32} diff {_a.mean() - _full.mean():+6.2f} pp   t({len(_a) - 1}) = {_t:6.2f}   p = {_p:.2g}")
